# Llama 3.1 · low-noise held-out evaluation
**One seed at a time: 123, then 456, then 789 · saved adapters · no training**

Run Installation, restart, then Sections 1–5 on an L4. Section 2 generates 5,000 greedy answers and 50,000 sampled answers per seed for the 1,000-question test split. Caches resume by question. Stop after Section 5 and send the blinded review ZIP. After restoring reviewed files, run Section 1 and Section 6. Repeat for the next seed.

Primary method: original all-member bidirectional NLI clustering, with empty samples retained in a separate no-answer group. Audited development corrections are a sensitivity analysis only. Each condition/seed uses its own original development threshold, embedded and frozen below. No threshold is fitted on test outcomes.

This split is held out from these Llama training/development runs, but was used in earlier Qwen experiments. Disclose that exposure; this is not an untouched independent final dataset. The noise range was selected after development experiments. Correctness labels remain AI-assisted unless reviewed by humans.

Outputs are separate from prior experiments. Preserve all generated outputs, including empty samples and truncations. Empty greedy answers trigger an investigation before scoring; they are saved and never silently resampled. All seeds must use identical question-validity decisions before aggregation.


## Installation · restart afterward

In [ ]:
%pip install -q "transformers==4.46.3" "tokenizers==0.20.3" "peft==0.13.2" "accelerate==1.1.1" "huggingface-hub==0.26.2" "safetensors==0.4.5" "sentencepiece==0.2.0" "scikit-learn==1.6.1" "numpy<3" "pandas<3" "matplotlib<4"
%pip uninstall -y torchao
# Restart session now; continue at Section 1.

## 1 · Freeze settings, thresholds and adapter identities
Only change SEED between complete runs. Keep TRAIN_VERSION and TEST_VERSION fixed unless intentionally starting a separately documented evaluation.


In [ ]:
from google.colab import drive
from pathlib import Path
import json,os,hashlib,gc,math,random,secrets,zipfile,time
import importlib.metadata as metadata
import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm
from IPython.display import display
drive.mount('/content/drive')
PROJECT=Path('/content/drive/MyDrive/dp_uq_experiment')
SEED=789
TRAIN_VERSION='v1'
TEST_VERSION='v1'
assert SEED in [123,456,789]
TRAIN=PROJECT/'checkpoints/llama31_8b_low_noise_pilot'/f'seed{SEED}_{TRAIN_VERSION}'
SHARED=PROJECT/'results'/f'llama31_8b_low_noise_test_{TEST_VERSION}'
OUT=SHARED/f'seed{SEED}_{TRAIN_VERSION}'
OUT.mkdir(parents=True,exist_ok=True)
PRE=PROJECT/'results/llama31_8b_preflight_v1'
CONDITIONS=['standard_poisson','clipping_poisson','dp_sigma_0p1','dp_sigma_0p25','dp_sigma_0p5']
SCORES=['mean_token_entropy','mean_answer_nll','discrete_semantic_entropy']
def read(p):return json.loads(p.read_text())
def save(v,p):
    t=p.with_suffix(p.suffix+'.tmp');t.write_text(json.dumps(v,ensure_ascii=False,indent=2,allow_nan=False));os.replace(t,p)
def fp(v):return hashlib.sha256(json.dumps(v,sort_keys=True,ensure_ascii=False).encode()).hexdigest()
def fh(p):
    h=hashlib.sha256()
    with open(p,'rb') as f:
        for b in iter(lambda:f.read(1024*1024),b''):h.update(b)
    return h.hexdigest()
def immutable(v,p):
    if p.exists():assert read(p)==v,f'Identity changed: {p}. Preserve existing files and investigate.'
    else:save(v,p)
def records_path(c):return OUT/(c+'_records.json')
def semantic_path(c):return OUT/(c+'_semantic.json')
FROZEN=json.loads('{"123": {"suite_config_hash": "31023399d743bf41ece4b39e893cda8a3925c11cded34a238caa9bec1086f00d", "thresholds": [{"condition": "standard_poisson", "score": "mean_token_entropy", "threshold": 1.0866200923919678, "abstain_all": false, "risk_target": 0.1, "selection_scope": "same development data; exploratory", "eligible_ids": ["odql_7980", "qg_749", "odql_11176", "sfq_20838", "sfq_2187", "bb_8725", "qw_10107", "odql_11899", "sfq_8726", "odql_10108", "odql_3325", "tb_1852", "qg_152", "sfq_4423", "sfq_10592", "sfq_18424", "odql_2963", "sfq_12277", "sfq_17015", "bb_9117", "qz_1730", "sfq_23986", "qf_2590", "jp_2935", "sfq_7702", "sfq_23359", "odql_13834", "qg_774", "odql_7981", "wh_2615", "qw_10089", "qw_4795", "dpql_4823", "qb_1953", "qb_292", "odql_9495", "odql_11469", "bb_3733", "odql_14999", "sfq_1917", "qg_527", "qf_3609", "sfq_22986", "sfq_21768", "qb_2880", "bb_2139", "qw_45", "qw_3576", "sfq_6988", "qb_2169", "qf_387", "sfq_3731", "tb_83", "qg_633", "wh_267", "odql_14829", "sfq_15164", "qw_4450", "qb_4066", "qb_3535", "bt_1378", "qg_504", "bt_61", "bb_7375", "bb_1588", "qb_3775", "sfq_15066", "dpql_782", "odql_1772", "qz_6456", "odql_8729", "qf_2569", "odql_8774", "bt_4062", "wh_4082", "qw_2489", "sfq_24884", "qb_8477", "dpql_1994", "odql_13686", "bb_5892", "qg_2735", "odql_2476", "bt_3047", "odql_8663", "qb_4792", "dpql_3600", "sfq_23513", "bt_4028", "sfq_4588", "dpql_2210", "bb_6465", "qb_38", "qz_4435", "odql_4457", "odql_9352", "sfq_19745", "bb_2691", "odql_6624", "qb_8289", "qw_12650", "bt_1819", "sfq_11945", "qz_2822", "qb_7094", "jp_4067", "sfq_6665", "bt_434", "odql_136", "odql_6494", "qz_1047", "qz_4231", "qw_6139", "bt_3299", "qf_277", "odql_249", "dpql_1447", "sfq_13640", "dpql_580", "qf_2048", "dpql_3467", "qb_3556", "qw_2843", "sfq_19473", "jp_4032", "qw_7594", "sfq_3202", "jp_3453", "qw_11112", "qf_345", "dpql_3635", "jp_2435", "odql_13482", "qw_1298", "sfq_15917", "odql_8493", "qg_357", "sfq_5739", "qg_1580", "odql_5501", "dpql_151", "qg_1800", "bb_7364", "qw_16265", "sfq_6175", "jp_1572", "qw_15027", "qz_6596", "odql_7507", "sfq_17997", "qb_3940", "qw_14565", "odql_7762", "qw_3207", "sfq_2355", "sfq_14806", "sfq_10687", "odql_8306", "sfq_16554", "odql_15094", "bb_6092", "qg_2424", "tb_299", "qb_9584", "sfq_7904", "sfq_10502", "tb_1911", "qg_3056", "bt_1811", "bt_371", "qz_5459", "qb_4163", "qb_8707", "odql_14527", "qg_2316", "dpql_1746", "qb_6635", "odql_2749", "odql_4794", "tc_719", "bt_1604", "qf_2264", "sfq_17840", "qb_2148", "dpql_5194", "tc_1944", "qw_15528", "odql_2965", "sfq_14668", "qw_10673", "dpql_4666", "jp_1502", "sfq_6752"]}, {"condition": "standard_poisson", "score": "mean_answer_nll", "threshold": 0.4078746736049652, "abstain_all": false, "risk_target": 0.1, "selection_scope": "same development data; exploratory", "eligible_ids": ["odql_7980", "qg_749", "odql_11176", "sfq_20838", "sfq_2187", "bb_8725", "qw_10107", "odql_11899", "sfq_8726", "odql_10108", "odql_3325", "tb_1852", "qg_152", "sfq_4423", "sfq_10592", "sfq_18424", "odql_2963", "sfq_12277", "sfq_17015", "bb_9117", "qz_1730", "sfq_23986", "qf_2590", "jp_2935", "sfq_7702", "sfq_23359", "odql_13834", "qg_774", "odql_7981", "wh_2615", "qw_10089", "qw_4795", "dpql_4823", "qb_1953", "qb_292", "odql_9495", "odql_11469", "bb_3733", "odql_14999", "sfq_1917", "qg_527", "qf_3609", "sfq_22986", "sfq_21768", "qb_2880", "bb_2139", "qw_45", "qw_3576", "sfq_6988", "qb_2169", "qf_387", "sfq_3731", "tb_83", "qg_633", "wh_267", "odql_14829", "sfq_15164", "qw_4450", "qb_4066", "qb_3535", "bt_1378", "qg_504", "bt_61", "bb_7375", "bb_1588", "qb_3775", "sfq_15066", "dpql_782", "odql_1772", "qz_6456", "odql_8729", "qf_2569", "odql_8774", "bt_4062", "wh_4082", "qw_2489", "sfq_24884", "qb_8477", "dpql_1994", "odql_13686", "bb_5892", "qg_2735", "odql_2476", "bt_3047", "odql_8663", "qb_4792", "dpql_3600", "sfq_23513", "bt_4028", "sfq_4588", "dpql_2210", "bb_6465", "qb_38", "qz_4435", "odql_4457", "odql_9352", "sfq_19745", "bb_2691", "odql_6624", "qb_8289", "qw_12650", "bt_1819", "sfq_11945", "qz_2822", "qb_7094", "jp_4067", "sfq_6665", "bt_434", "odql_136", "odql_6494", "qz_1047", "qz_4231", "qw_6139", "bt_3299", "qf_277", "odql_249", "dpql_1447", "sfq_13640", "dpql_580", "qf_2048", "dpql_3467", "qb_3556", "qw_2843", "sfq_19473", "jp_4032", "qw_7594", "sfq_3202", "jp_3453", "qw_11112", "qf_345", "dpql_3635", "jp_2435", "odql_13482", "qw_1298", "sfq_15917", "odql_8493", "qg_357", "sfq_5739", "qg_1580", "odql_5501", "dpql_151", "qg_1800", "bb_7364", "qw_16265", "sfq_6175", "jp_1572", "qw_15027", "qz_6596", "odql_7507", "sfq_17997", "qb_3940", "qw_14565", "odql_7762", "qw_3207", "sfq_2355", "sfq_14806", "sfq_10687", "odql_8306", "sfq_16554", "odql_15094", "bb_6092", "qg_2424", "tb_299", "qb_9584", "sfq_7904", "sfq_10502", "tb_1911", "qg_3056", "bt_1811", "bt_371", "qz_5459", "qb_4163", "qb_8707", "odql_14527", "qg_2316", "dpql_1746", "qb_6635", "odql_2749", "odql_4794", "tc_719", "bt_1604", "qf_2264", "sfq_17840", "qb_2148", "dpql_5194", "tc_1944", "qw_15528", "odql_2965", "sfq_14668", "qw_10673", "dpql_4666", "jp_1502", "sfq_6752"]}, {"condition": "standard_poisson", "score": "discrete_semantic_entropy", "threshold": 0.6931471805599453, "abstain_all": false, "risk_target": 0.1, "selection_scope": "same development data; exploratory", "eligible_ids": ["odql_7980", "qg_749", "odql_11176", "sfq_20838", "sfq_2187", "bb_8725", "qw_10107", "odql_11899", "sfq_8726", "odql_10108", "odql_3325", "tb_1852", "qg_152", "sfq_4423", "sfq_10592", "sfq_18424", "odql_2963", "sfq_12277", "sfq_17015", "bb_9117", "qz_1730", "sfq_23986", "qf_2590", "jp_2935", "sfq_7702", "sfq_23359", "odql_13834", "qg_774", "odql_7981", "wh_2615", "qw_10089", "qw_4795", "dpql_4823", "qb_1953", "qb_292", "odql_9495", "odql_11469", "bb_3733", "odql_14999", "sfq_1917", "qg_527", "qf_3609", "sfq_22986", "sfq_21768", "qb_2880", "bb_2139", "qw_45", "qw_3576", "sfq_6988", "qb_2169", "qf_387", "sfq_3731", "tb_83", "qg_633", "wh_267", "odql_14829", "sfq_15164", "qw_4450", "qb_4066", "qb_3535", "bt_1378", "qg_504", "bt_61", "bb_7375", "bb_1588", "qb_3775", "sfq_15066", "dpql_782", "odql_1772", "qz_6456", "odql_8729", "qf_2569", "odql_8774", "bt_4062", "wh_4082", "qw_2489", "sfq_24884", "qb_8477", "dpql_1994", "odql_13686", "bb_5892", "qg_2735", "odql_2476", "bt_3047", "odql_8663", "qb_4792", "dpql_3600", "sfq_23513", "bt_4028", "sfq_4588", "dpql_2210", "bb_6465", "qb_38", "qz_4435", "odql_4457", "odql_9352", "sfq_19745", "bb_2691", "odql_6624", "qb_8289", "qw_12650", "bt_1819", "sfq_11945", "qz_2822", "qb_7094", "jp_4067", "sfq_6665", "bt_434", "odql_136", "odql_6494", "qz_1047", "qz_4231", "qw_6139", "bt_3299", "qf_277", "odql_249", "dpql_1447", "sfq_13640", "dpql_580", "qf_2048", "dpql_3467", "qb_3556", "qw_2843", "sfq_19473", "jp_4032", "qw_7594", "sfq_3202", "jp_3453", "qw_11112", "qf_345", "dpql_3635", "jp_2435", "odql_13482", "qw_1298", "sfq_15917", "odql_8493", "qg_357", "sfq_5739", "qg_1580", "odql_5501", "dpql_151", "qg_1800", "bb_7364", "qw_16265", "sfq_6175", "jp_1572", "qw_15027", "qz_6596", "odql_7507", "sfq_17997", "qb_3940", "qw_14565", "odql_7762", "qw_3207", "sfq_2355", "sfq_14806", "sfq_10687", "odql_8306", "sfq_16554", "odql_15094", "bb_6092", "qg_2424", "tb_299", "qb_9584", "sfq_7904", "sfq_10502", "tb_1911", "qg_3056", "bt_1811", "bt_371", "qz_5459", "qb_4163", "qb_8707", "odql_14527", "qg_2316", "dpql_1746", "qb_6635", "odql_2749", "odql_4794", "tc_719", "bt_1604", "qf_2264", "sfq_17840", "qb_2148", "dpql_5194", "tc_1944", "qw_15528", "odql_2965", "sfq_14668", "qw_10673", "dpql_4666", "jp_1502", "sfq_6752"]}, {"condition": "clipping_poisson", "score": "mean_token_entropy", "threshold": 0.96396803855896, "abstain_all": false, "risk_target": 0.1, "selection_scope": "same development data; exploratory", "eligible_ids": ["odql_7980", "qg_749", "odql_11176", "sfq_20838", "sfq_2187", "bb_8725", "qw_10107", "odql_11899", "sfq_8726", "odql_10108", "odql_3325", "tb_1852", "qg_152", "sfq_4423", "sfq_10592", "sfq_18424", "odql_2963", "sfq_12277", "sfq_17015", "bb_9117", "qz_1730", "sfq_23986", "qf_2590", "jp_2935", "sfq_7702", "sfq_23359", "odql_13834", "qg_774", "odql_7981", "wh_2615", "qw_10089", "qw_4795", "dpql_4823", "qb_1953", "qb_292", "odql_9495", "odql_11469", "bb_3733", "odql_14999", "sfq_1917", "qg_527", "qf_3609", "sfq_22986", "sfq_21768", "qb_2880", "bb_2139", "qw_45", "qw_3576", "sfq_6988", "qb_2169", "qf_387", "sfq_3731", "tb_83", "qg_633", "wh_267", "odql_14829", "sfq_15164", "qw_4450", "qb_4066", "qb_3535", "bt_1378", "qg_504", "bt_61", "bb_7375", "bb_1588", "qb_3775", "sfq_15066", "dpql_782", "odql_1772", "qz_6456", "odql_8729", "qf_2569", "odql_8774", "bt_4062", "wh_4082", "qw_2489", "sfq_24884", "qb_8477", "dpql_1994", "odql_13686", "bb_5892", "qg_2735", "odql_2476", "bt_3047", "odql_8663", "qb_4792", "dpql_3600", "sfq_23513", "bt_4028", "sfq_4588", "dpql_2210", "bb_6465", "qb_38", "qz_4435", "odql_4457", "odql_9352", "sfq_19745", "bb_2691", "odql_6624", "qb_8289", "qw_12650", "bt_1819", "sfq_11945", "qz_2822", "qb_7094", "jp_4067", "sfq_6665", "bt_434", "odql_136", "odql_6494", "qz_1047", "qz_4231", "qw_6139", "bt_3299", "qf_277", "odql_249", "dpql_1447", "sfq_13640", "dpql_580", "qf_2048", "dpql_3467", "qb_3556", "qw_2843", "sfq_19473", "jp_4032", "qw_7594", "sfq_3202", "jp_3453", "qw_11112", "qf_345", "dpql_3635", "jp_2435", "odql_13482", "qw_1298", "sfq_15917", "odql_8493", "qg_357", "sfq_5739", "qg_1580", "odql_5501", "dpql_151", "qg_1800", "bb_7364", "qw_16265", "sfq_6175", "jp_1572", "qw_15027", "qz_6596", "odql_7507", "sfq_17997", "qb_3940", "qw_14565", "odql_7762", "qw_3207", "sfq_2355", "sfq_14806", "sfq_10687", "odql_8306", "sfq_16554", "odql_15094", "bb_6092", "qg_2424", "tb_299", "qb_9584", "sfq_7904", "sfq_10502", "tb_1911", "qg_3056", "bt_1811", "bt_371", "qz_5459", "qb_4163", "qb_8707", "odql_14527", "qg_2316", "dpql_1746", "qb_6635", "odql_2749", "odql_4794", "tc_719", "bt_1604", "qf_2264", "sfq_17840", "qb_2148", "dpql_5194", "tc_1944", "qw_15528", "odql_2965", "sfq_14668", "qw_10673", "dpql_4666", "jp_1502", "sfq_6752"]}, {"condition": "clipping_poisson", "score": "mean_answer_nll", "threshold": 0.31850719451904297, "abstain_all": false, "risk_target": 0.1, "selection_scope": "same development data; exploratory", "eligible_ids": ["odql_7980", "qg_749", "odql_11176", "sfq_20838", "sfq_2187", "bb_8725", "qw_10107", "odql_11899", "sfq_8726", "odql_10108", "odql_3325", "tb_1852", "qg_152", "sfq_4423", "sfq_10592", "sfq_18424", "odql_2963", "sfq_12277", "sfq_17015", "bb_9117", "qz_1730", "sfq_23986", "qf_2590", "jp_2935", "sfq_7702", "sfq_23359", "odql_13834", "qg_774", "odql_7981", "wh_2615", "qw_10089", "qw_4795", "dpql_4823", "qb_1953", "qb_292", "odql_9495", "odql_11469", "bb_3733", "odql_14999", "sfq_1917", "qg_527", "qf_3609", "sfq_22986", "sfq_21768", "qb_2880", "bb_2139", "qw_45", "qw_3576", "sfq_6988", "qb_2169", "qf_387", "sfq_3731", "tb_83", "qg_633", "wh_267", "odql_14829", "sfq_15164", "qw_4450", "qb_4066", "qb_3535", "bt_1378", "qg_504", "bt_61", "bb_7375", "bb_1588", "qb_3775", "sfq_15066", "dpql_782", "odql_1772", "qz_6456", "odql_8729", "qf_2569", "odql_8774", "bt_4062", "wh_4082", "qw_2489", "sfq_24884", "qb_8477", "dpql_1994", "odql_13686", "bb_5892", "qg_2735", "odql_2476", "bt_3047", "odql_8663", "qb_4792", "dpql_3600", "sfq_23513", "bt_4028", "sfq_4588", "dpql_2210", "bb_6465", "qb_38", "qz_4435", "odql_4457", "odql_9352", "sfq_19745", "bb_2691", "odql_6624", "qb_8289", "qw_12650", "bt_1819", "sfq_11945", "qz_2822", "qb_7094", "jp_4067", "sfq_6665", "bt_434", "odql_136", "odql_6494", "qz_1047", "qz_4231", "qw_6139", "bt_3299", "qf_277", "odql_249", "dpql_1447", "sfq_13640", "dpql_580", "qf_2048", "dpql_3467", "qb_3556", "qw_2843", "sfq_19473", "jp_4032", "qw_7594", "sfq_3202", "jp_3453", "qw_11112", "qf_345", "dpql_3635", "jp_2435", "odql_13482", "qw_1298", "sfq_15917", "odql_8493", "qg_357", "sfq_5739", "qg_1580", "odql_5501", "dpql_151", "qg_1800", "bb_7364", "qw_16265", "sfq_6175", "jp_1572", "qw_15027", "qz_6596", "odql_7507", "sfq_17997", "qb_3940", "qw_14565", "odql_7762", "qw_3207", "sfq_2355", "sfq_14806", "sfq_10687", "odql_8306", "sfq_16554", "odql_15094", "bb_6092", "qg_2424", "tb_299", "qb_9584", "sfq_7904", "sfq_10502", "tb_1911", "qg_3056", "bt_1811", "bt_371", "qz_5459", "qb_4163", "qb_8707", "odql_14527", "qg_2316", "dpql_1746", "qb_6635", "odql_2749", "odql_4794", "tc_719", "bt_1604", "qf_2264", "sfq_17840", "qb_2148", "dpql_5194", "tc_1944", "qw_15528", "odql_2965", "sfq_14668", "qw_10673", "dpql_4666", "jp_1502", "sfq_6752"]}, {"condition": "clipping_poisson", "score": "discrete_semantic_entropy", "threshold": 0.3250829733914482, "abstain_all": false, "risk_target": 0.1, "selection_scope": "same development data; exploratory", "eligible_ids": ["odql_7980", "qg_749", "odql_11176", "sfq_20838", "sfq_2187", "bb_8725", "qw_10107", "odql_11899", "sfq_8726", "odql_10108", "odql_3325", "tb_1852", "qg_152", "sfq_4423", "sfq_10592", "sfq_18424", "odql_2963", "sfq_12277", "sfq_17015", "bb_9117", "qz_1730", "sfq_23986", "qf_2590", "jp_2935", "sfq_7702", "sfq_23359", "odql_13834", "qg_774", "odql_7981", "wh_2615", "qw_10089", "qw_4795", "dpql_4823", "qb_1953", "qb_292", "odql_9495", "odql_11469", "bb_3733", "odql_14999", "sfq_1917", "qg_527", "qf_3609", "sfq_22986", "sfq_21768", "qb_2880", "bb_2139", "qw_45", "qw_3576", "sfq_6988", "qb_2169", "qf_387", "sfq_3731", "tb_83", "qg_633", "wh_267", "odql_14829", "sfq_15164", "qw_4450", "qb_4066", "qb_3535", "bt_1378", "qg_504", "bt_61", "bb_7375", "bb_1588", "qb_3775", "sfq_15066", "dpql_782", "odql_1772", "qz_6456", "odql_8729", "qf_2569", "odql_8774", "bt_4062", "wh_4082", "qw_2489", "sfq_24884", "qb_8477", "dpql_1994", "odql_13686", "bb_5892", "qg_2735", "odql_2476", "bt_3047", "odql_8663", "qb_4792", "dpql_3600", "sfq_23513", "bt_4028", "sfq_4588", "dpql_2210", "bb_6465", "qb_38", "qz_4435", "odql_4457", "odql_9352", "sfq_19745", "bb_2691", "odql_6624", "qb_8289", "qw_12650", "bt_1819", "sfq_11945", "qz_2822", "qb_7094", "jp_4067", "sfq_6665", "bt_434", "odql_136", "odql_6494", "qz_1047", "qz_4231", "qw_6139", "bt_3299", "qf_277", "odql_249", "dpql_1447", "sfq_13640", "dpql_580", "qf_2048", "dpql_3467", "qb_3556", "qw_2843", "sfq_19473", "jp_4032", "qw_7594", "sfq_3202", "jp_3453", "qw_11112", "qf_345", "dpql_3635", "jp_2435", "odql_13482", "qw_1298", "sfq_15917", "odql_8493", "qg_357", "sfq_5739", "qg_1580", "odql_5501", "dpql_151", "qg_1800", "bb_7364", "qw_16265", "sfq_6175", "jp_1572", "qw_15027", "qz_6596", "odql_7507", "sfq_17997", "qb_3940", "qw_14565", "odql_7762", "qw_3207", "sfq_2355", "sfq_14806", "sfq_10687", "odql_8306", "sfq_16554", "odql_15094", "bb_6092", "qg_2424", "tb_299", "qb_9584", "sfq_7904", "sfq_10502", "tb_1911", "qg_3056", "bt_1811", "bt_371", "qz_5459", "qb_4163", "qb_8707", "odql_14527", "qg_2316", "dpql_1746", "qb_6635", "odql_2749", "odql_4794", "tc_719", "bt_1604", "qf_2264", "sfq_17840", "qb_2148", "dpql_5194", "tc_1944", "qw_15528", "odql_2965", "sfq_14668", "qw_10673", "dpql_4666", "jp_1502", "sfq_6752"]}, {"condition": "dp_sigma_0p1", "score": "mean_token_entropy", "threshold": 1.193043828010559, "abstain_all": false, "risk_target": 0.1, "selection_scope": "same development data; exploratory", "eligible_ids": ["odql_7980", "qg_749", "odql_11176", "sfq_20838", "sfq_2187", "bb_8725", "qw_10107", "odql_11899", "sfq_8726", "odql_10108", "odql_3325", "tb_1852", "qg_152", "sfq_4423", "sfq_10592", "sfq_18424", "odql_2963", "sfq_12277", "sfq_17015", "bb_9117", "qz_1730", "sfq_23986", "qf_2590", "jp_2935", "sfq_7702", "sfq_23359", "odql_13834", "qg_774", "odql_7981", "wh_2615", "qw_10089", "qw_4795", "dpql_4823", "qb_1953", "qb_292", "odql_9495", "odql_11469", "bb_3733", "odql_14999", "sfq_1917", "qg_527", "qf_3609", "sfq_22986", "sfq_21768", "qb_2880", "bb_2139", "qw_45", "qw_3576", "sfq_6988", "qb_2169", "qf_387", "sfq_3731", "tb_83", "qg_633", "wh_267", "odql_14829", "sfq_15164", "qw_4450", "qb_4066", "qb_3535", "bt_1378", "qg_504", "bt_61", "bb_7375", "bb_1588", "qb_3775", "sfq_15066", "dpql_782", "odql_1772", "qz_6456", "odql_8729", "qf_2569", "odql_8774", "bt_4062", "wh_4082", "qw_2489", "sfq_24884", "qb_8477", "dpql_1994", "odql_13686", "bb_5892", "qg_2735", "odql_2476", "bt_3047", "odql_8663", "qb_4792", "dpql_3600", "sfq_23513", "bt_4028", "sfq_4588", "dpql_2210", "bb_6465", "qb_38", "qz_4435", "odql_4457", "odql_9352", "sfq_19745", "bb_2691", "odql_6624", "qb_8289", "qw_12650", "bt_1819", "sfq_11945", "qz_2822", "qb_7094", "jp_4067", "sfq_6665", "bt_434", "odql_136", "odql_6494", "qz_1047", "qz_4231", "qw_6139", "bt_3299", "qf_277", "odql_249", "dpql_1447", "sfq_13640", "dpql_580", "qf_2048", "dpql_3467", "qb_3556", "qw_2843", "sfq_19473", "jp_4032", "qw_7594", "sfq_3202", "jp_3453", "qw_11112", "qf_345", "dpql_3635", "jp_2435", "odql_13482", "qw_1298", "sfq_15917", "odql_8493", "qg_357", "sfq_5739", "qg_1580", "odql_5501", "dpql_151", "qg_1800", "bb_7364", "qw_16265", "sfq_6175", "jp_1572", "qw_15027", "qz_6596", "odql_7507", "sfq_17997", "qb_3940", "qw_14565", "odql_7762", "qw_3207", "sfq_2355", "sfq_14806", "sfq_10687", "odql_8306", "sfq_16554", "odql_15094", "bb_6092", "qg_2424", "tb_299", "qb_9584", "sfq_7904", "sfq_10502", "tb_1911", "qg_3056", "bt_1811", "bt_371", "qz_5459", "qb_4163", "qb_8707", "odql_14527", "qg_2316", "dpql_1746", "qb_6635", "odql_2749", "odql_4794", "tc_719", "bt_1604", "qf_2264", "sfq_17840", "qb_2148", "dpql_5194", "tc_1944", "qw_15528", "odql_2965", "sfq_14668", "qw_10673", "dpql_4666", "jp_1502", "sfq_6752"]}, {"condition": "dp_sigma_0p1", "score": "mean_answer_nll", "threshold": 0.4842994213104248, "abstain_all": false, "risk_target": 0.1, "selection_scope": "same development data; exploratory", "eligible_ids": ["odql_7980", "qg_749", "odql_11176", "sfq_20838", "sfq_2187", "bb_8725", "qw_10107", "odql_11899", "sfq_8726", "odql_10108", "odql_3325", "tb_1852", "qg_152", "sfq_4423", "sfq_10592", "sfq_18424", "odql_2963", "sfq_12277", "sfq_17015", "bb_9117", "qz_1730", "sfq_23986", "qf_2590", "jp_2935", "sfq_7702", "sfq_23359", "odql_13834", "qg_774", "odql_7981", "wh_2615", "qw_10089", "qw_4795", "dpql_4823", "qb_1953", "qb_292", "odql_9495", "odql_11469", "bb_3733", "odql_14999", "sfq_1917", "qg_527", "qf_3609", "sfq_22986", "sfq_21768", "qb_2880", "bb_2139", "qw_45", "qw_3576", "sfq_6988", "qb_2169", "qf_387", "sfq_3731", "tb_83", "qg_633", "wh_267", "odql_14829", "sfq_15164", "qw_4450", "qb_4066", "qb_3535", "bt_1378", "qg_504", "bt_61", "bb_7375", "bb_1588", "qb_3775", "sfq_15066", "dpql_782", "odql_1772", "qz_6456", "odql_8729", "qf_2569", "odql_8774", "bt_4062", "wh_4082", "qw_2489", "sfq_24884", "qb_8477", "dpql_1994", "odql_13686", "bb_5892", "qg_2735", "odql_2476", "bt_3047", "odql_8663", "qb_4792", "dpql_3600", "sfq_23513", "bt_4028", "sfq_4588", "dpql_2210", "bb_6465", "qb_38", "qz_4435", "odql_4457", "odql_9352", "sfq_19745", "bb_2691", "odql_6624", "qb_8289", "qw_12650", "bt_1819", "sfq_11945", "qz_2822", "qb_7094", "jp_4067", "sfq_6665", "bt_434", "odql_136", "odql_6494", "qz_1047", "qz_4231", "qw_6139", "bt_3299", "qf_277", "odql_249", "dpql_1447", "sfq_13640", "dpql_580", "qf_2048", "dpql_3467", "qb_3556", "qw_2843", "sfq_19473", "jp_4032", "qw_7594", "sfq_3202", "jp_3453", "qw_11112", "qf_345", "dpql_3635", "jp_2435", "odql_13482", "qw_1298", "sfq_15917", "odql_8493", "qg_357", "sfq_5739", "qg_1580", "odql_5501", "dpql_151", "qg_1800", "bb_7364", "qw_16265", "sfq_6175", "jp_1572", "qw_15027", "qz_6596", "odql_7507", "sfq_17997", "qb_3940", "qw_14565", "odql_7762", "qw_3207", "sfq_2355", "sfq_14806", "sfq_10687", "odql_8306", "sfq_16554", "odql_15094", "bb_6092", "qg_2424", "tb_299", "qb_9584", "sfq_7904", "sfq_10502", "tb_1911", "qg_3056", "bt_1811", "bt_371", "qz_5459", "qb_4163", "qb_8707", "odql_14527", "qg_2316", "dpql_1746", "qb_6635", "odql_2749", "odql_4794", "tc_719", "bt_1604", "qf_2264", "sfq_17840", "qb_2148", "dpql_5194", "tc_1944", "qw_15528", "odql_2965", "sfq_14668", "qw_10673", "dpql_4666", "jp_1502", "sfq_6752"]}, {"condition": "dp_sigma_0p1", "score": "discrete_semantic_entropy", "threshold": 0.8018185525433373, "abstain_all": false, "risk_target": 0.1, "selection_scope": "same development data; exploratory", "eligible_ids": ["odql_7980", "qg_749", "odql_11176", "sfq_20838", "sfq_2187", "bb_8725", "qw_10107", "odql_11899", "sfq_8726", "odql_10108", "odql_3325", "tb_1852", "qg_152", "sfq_4423", "sfq_10592", "sfq_18424", "odql_2963", "sfq_12277", "sfq_17015", "bb_9117", "qz_1730", "sfq_23986", "qf_2590", "jp_2935", "sfq_7702", "sfq_23359", "odql_13834", "qg_774", "odql_7981", "wh_2615", "qw_10089", "qw_4795", "dpql_4823", "qb_1953", "qb_292", "odql_9495", "odql_11469", "bb_3733", "odql_14999", "sfq_1917", "qg_527", "qf_3609", "sfq_22986", "sfq_21768", "qb_2880", "bb_2139", "qw_45", "qw_3576", "sfq_6988", "qb_2169", "qf_387", "sfq_3731", "tb_83", "qg_633", "wh_267", "odql_14829", "sfq_15164", "qw_4450", "qb_4066", "qb_3535", "bt_1378", "qg_504", "bt_61", "bb_7375", "bb_1588", "qb_3775", "sfq_15066", "dpql_782", "odql_1772", "qz_6456", "odql_8729", "qf_2569", "odql_8774", "bt_4062", "wh_4082", "qw_2489", "sfq_24884", "qb_8477", "dpql_1994", "odql_13686", "bb_5892", "qg_2735", "odql_2476", "bt_3047", "odql_8663", "qb_4792", "dpql_3600", "sfq_23513", "bt_4028", "sfq_4588", "dpql_2210", "bb_6465", "qb_38", "qz_4435", "odql_4457", "odql_9352", "sfq_19745", "bb_2691", "odql_6624", "qb_8289", "qw_12650", "bt_1819", "sfq_11945", "qz_2822", "qb_7094", "jp_4067", "sfq_6665", "bt_434", "odql_136", "odql_6494", "qz_1047", "qz_4231", "qw_6139", "bt_3299", "qf_277", "odql_249", "dpql_1447", "sfq_13640", "dpql_580", "qf_2048", "dpql_3467", "qb_3556", "qw_2843", "sfq_19473", "jp_4032", "qw_7594", "sfq_3202", "jp_3453", "qw_11112", "qf_345", "dpql_3635", "jp_2435", "odql_13482", "qw_1298", "sfq_15917", "odql_8493", "qg_357", "sfq_5739", "qg_1580", "odql_5501", "dpql_151", "qg_1800", "bb_7364", "qw_16265", "sfq_6175", "jp_1572", "qw_15027", "qz_6596", "odql_7507", "sfq_17997", "qb_3940", "qw_14565", "odql_7762", "qw_3207", "sfq_2355", "sfq_14806", "sfq_10687", "odql_8306", "sfq_16554", "odql_15094", "bb_6092", "qg_2424", "tb_299", "qb_9584", "sfq_7904", "sfq_10502", "tb_1911", "qg_3056", "bt_1811", "bt_371", "qz_5459", "qb_4163", "qb_8707", "odql_14527", "qg_2316", "dpql_1746", "qb_6635", "odql_2749", "odql_4794", "tc_719", "bt_1604", "qf_2264", "sfq_17840", "qb_2148", "dpql_5194", "tc_1944", "qw_15528", "odql_2965", "sfq_14668", "qw_10673", "dpql_4666", "jp_1502", "sfq_6752"]}, {"condition": "dp_sigma_0p25", "score": "mean_token_entropy", "threshold": 1.0105692148208618, "abstain_all": false, "risk_target": 0.1, "selection_scope": "same development data; exploratory", "eligible_ids": ["odql_7980", "qg_749", "odql_11176", "sfq_20838", "sfq_2187", "bb_8725", "qw_10107", "odql_11899", "sfq_8726", "odql_10108", "odql_3325", "tb_1852", "qg_152", "sfq_4423", "sfq_10592", "sfq_18424", "odql_2963", "sfq_12277", "sfq_17015", "bb_9117", "qz_1730", "sfq_23986", "qf_2590", "jp_2935", "sfq_7702", "sfq_23359", "odql_13834", "qg_774", "odql_7981", "wh_2615", "qw_10089", "qw_4795", "dpql_4823", "qb_1953", "qb_292", "odql_9495", "odql_11469", "bb_3733", "odql_14999", "sfq_1917", "qg_527", "qf_3609", "sfq_22986", "sfq_21768", "qb_2880", "bb_2139", "qw_45", "qw_3576", "sfq_6988", "qb_2169", "qf_387", "sfq_3731", "tb_83", "qg_633", "wh_267", "odql_14829", "sfq_15164", "qw_4450", "qb_4066", "qb_3535", "bt_1378", "qg_504", "bt_61", "bb_7375", "bb_1588", "qb_3775", "sfq_15066", "dpql_782", "odql_1772", "qz_6456", "odql_8729", "qf_2569", "odql_8774", "bt_4062", "wh_4082", "qw_2489", "sfq_24884", "qb_8477", "dpql_1994", "odql_13686", "bb_5892", "qg_2735", "odql_2476", "bt_3047", "odql_8663", "qb_4792", "dpql_3600", "sfq_23513", "bt_4028", "sfq_4588", "dpql_2210", "bb_6465", "qb_38", "qz_4435", "odql_4457", "odql_9352", "sfq_19745", "bb_2691", "odql_6624", "qb_8289", "qw_12650", "bt_1819", "sfq_11945", "qz_2822", "qb_7094", "jp_4067", "sfq_6665", "bt_434", "odql_136", "odql_6494", "qz_1047", "qz_4231", "qw_6139", "bt_3299", "qf_277", "odql_249", "dpql_1447", "sfq_13640", "dpql_580", "qf_2048", "dpql_3467", "qb_3556", "qw_2843", "sfq_19473", "jp_4032", "qw_7594", "sfq_3202", "jp_3453", "qw_11112", "qf_345", "dpql_3635", "jp_2435", "odql_13482", "qw_1298", "sfq_15917", "odql_8493", "qg_357", "sfq_5739", "qg_1580", "odql_5501", "dpql_151", "qg_1800", "bb_7364", "qw_16265", "sfq_6175", "jp_1572", "qw_15027", "qz_6596", "odql_7507", "sfq_17997", "qb_3940", "qw_14565", "odql_7762", "qw_3207", "sfq_2355", "sfq_14806", "sfq_10687", "odql_8306", "sfq_16554", "odql_15094", "bb_6092", "qg_2424", "tb_299", "qb_9584", "sfq_7904", "sfq_10502", "tb_1911", "qg_3056", "bt_1811", "bt_371", "qz_5459", "qb_4163", "qb_8707", "odql_14527", "qg_2316", "dpql_1746", "qb_6635", "odql_2749", "odql_4794", "tc_719", "bt_1604", "qf_2264", "sfq_17840", "qb_2148", "dpql_5194", "tc_1944", "qw_15528", "odql_2965", "sfq_14668", "qw_10673", "dpql_4666", "jp_1502", "sfq_6752"]}, {"condition": "dp_sigma_0p25", "score": "mean_answer_nll", "threshold": 0.4213770031929016, "abstain_all": false, "risk_target": 0.1, "selection_scope": "same development data; exploratory", "eligible_ids": ["odql_7980", "qg_749", "odql_11176", "sfq_20838", "sfq_2187", "bb_8725", "qw_10107", "odql_11899", "sfq_8726", "odql_10108", "odql_3325", "tb_1852", "qg_152", "sfq_4423", "sfq_10592", "sfq_18424", "odql_2963", "sfq_12277", "sfq_17015", "bb_9117", "qz_1730", "sfq_23986", "qf_2590", "jp_2935", "sfq_7702", "sfq_23359", "odql_13834", "qg_774", "odql_7981", "wh_2615", "qw_10089", "qw_4795", "dpql_4823", "qb_1953", "qb_292", "odql_9495", "odql_11469", "bb_3733", "odql_14999", "sfq_1917", "qg_527", "qf_3609", "sfq_22986", "sfq_21768", "qb_2880", "bb_2139", "qw_45", "qw_3576", "sfq_6988", "qb_2169", "qf_387", "sfq_3731", "tb_83", "qg_633", "wh_267", "odql_14829", "sfq_15164", "qw_4450", "qb_4066", "qb_3535", "bt_1378", "qg_504", "bt_61", "bb_7375", "bb_1588", "qb_3775", "sfq_15066", "dpql_782", "odql_1772", "qz_6456", "odql_8729", "qf_2569", "odql_8774", "bt_4062", "wh_4082", "qw_2489", "sfq_24884", "qb_8477", "dpql_1994", "odql_13686", "bb_5892", "qg_2735", "odql_2476", "bt_3047", "odql_8663", "qb_4792", "dpql_3600", "sfq_23513", "bt_4028", "sfq_4588", "dpql_2210", "bb_6465", "qb_38", "qz_4435", "odql_4457", "odql_9352", "sfq_19745", "bb_2691", "odql_6624", "qb_8289", "qw_12650", "bt_1819", "sfq_11945", "qz_2822", "qb_7094", "jp_4067", "sfq_6665", "bt_434", "odql_136", "odql_6494", "qz_1047", "qz_4231", "qw_6139", "bt_3299", "qf_277", "odql_249", "dpql_1447", "sfq_13640", "dpql_580", "qf_2048", "dpql_3467", "qb_3556", "qw_2843", "sfq_19473", "jp_4032", "qw_7594", "sfq_3202", "jp_3453", "qw_11112", "qf_345", "dpql_3635", "jp_2435", "odql_13482", "qw_1298", "sfq_15917", "odql_8493", "qg_357", "sfq_5739", "qg_1580", "odql_5501", "dpql_151", "qg_1800", "bb_7364", "qw_16265", "sfq_6175", "jp_1572", "qw_15027", "qz_6596", "odql_7507", "sfq_17997", "qb_3940", "qw_14565", "odql_7762", "qw_3207", "sfq_2355", "sfq_14806", "sfq_10687", "odql_8306", "sfq_16554", "odql_15094", "bb_6092", "qg_2424", "tb_299", "qb_9584", "sfq_7904", "sfq_10502", "tb_1911", "qg_3056", "bt_1811", "bt_371", "qz_5459", "qb_4163", "qb_8707", "odql_14527", "qg_2316", "dpql_1746", "qb_6635", "odql_2749", "odql_4794", "tc_719", "bt_1604", "qf_2264", "sfq_17840", "qb_2148", "dpql_5194", "tc_1944", "qw_15528", "odql_2965", "sfq_14668", "qw_10673", "dpql_4666", "jp_1502", "sfq_6752"]}, {"condition": "dp_sigma_0p25", "score": "discrete_semantic_entropy", "threshold": 0.6108643020548935, "abstain_all": false, "risk_target": 0.1, "selection_scope": "same development data; exploratory", "eligible_ids": ["odql_7980", "qg_749", "odql_11176", "sfq_20838", "sfq_2187", "bb_8725", "qw_10107", "odql_11899", "sfq_8726", "odql_10108", "odql_3325", "tb_1852", "qg_152", "sfq_4423", "sfq_10592", "sfq_18424", "odql_2963", "sfq_12277", "sfq_17015", "bb_9117", "qz_1730", "sfq_23986", "qf_2590", "jp_2935", "sfq_7702", "sfq_23359", "odql_13834", "qg_774", "odql_7981", "wh_2615", "qw_10089", "qw_4795", "dpql_4823", "qb_1953", "qb_292", "odql_9495", "odql_11469", "bb_3733", "odql_14999", "sfq_1917", "qg_527", "qf_3609", "sfq_22986", "sfq_21768", "qb_2880", "bb_2139", "qw_45", "qw_3576", "sfq_6988", "qb_2169", "qf_387", "sfq_3731", "tb_83", "qg_633", "wh_267", "odql_14829", "sfq_15164", "qw_4450", "qb_4066", "qb_3535", "bt_1378", "qg_504", "bt_61", "bb_7375", "bb_1588", "qb_3775", "sfq_15066", "dpql_782", "odql_1772", "qz_6456", "odql_8729", "qf_2569", "odql_8774", "bt_4062", "wh_4082", "qw_2489", "sfq_24884", "qb_8477", "dpql_1994", "odql_13686", "bb_5892", "qg_2735", "odql_2476", "bt_3047", "odql_8663", "qb_4792", "dpql_3600", "sfq_23513", "bt_4028", "sfq_4588", "dpql_2210", "bb_6465", "qb_38", "qz_4435", "odql_4457", "odql_9352", "sfq_19745", "bb_2691", "odql_6624", "qb_8289", "qw_12650", "bt_1819", "sfq_11945", "qz_2822", "qb_7094", "jp_4067", "sfq_6665", "bt_434", "odql_136", "odql_6494", "qz_1047", "qz_4231", "qw_6139", "bt_3299", "qf_277", "odql_249", "dpql_1447", "sfq_13640", "dpql_580", "qf_2048", "dpql_3467", "qb_3556", "qw_2843", "sfq_19473", "jp_4032", "qw_7594", "sfq_3202", "jp_3453", "qw_11112", "qf_345", "dpql_3635", "jp_2435", "odql_13482", "qw_1298", "sfq_15917", "odql_8493", "qg_357", "sfq_5739", "qg_1580", "odql_5501", "dpql_151", "qg_1800", "bb_7364", "qw_16265", "sfq_6175", "jp_1572", "qw_15027", "qz_6596", "odql_7507", "sfq_17997", "qb_3940", "qw_14565", "odql_7762", "qw_3207", "sfq_2355", "sfq_14806", "sfq_10687", "odql_8306", "sfq_16554", "odql_15094", "bb_6092", "qg_2424", "tb_299", "qb_9584", "sfq_7904", "sfq_10502", "tb_1911", "qg_3056", "bt_1811", "bt_371", "qz_5459", "qb_4163", "qb_8707", "odql_14527", "qg_2316", "dpql_1746", "qb_6635", "odql_2749", "odql_4794", "tc_719", "bt_1604", "qf_2264", "sfq_17840", "qb_2148", "dpql_5194", "tc_1944", "qw_15528", "odql_2965", "sfq_14668", "qw_10673", "dpql_4666", "jp_1502", "sfq_6752"]}, {"condition": "dp_sigma_0p5", "score": "mean_token_entropy", "threshold": 0.9846554398536682, "abstain_all": false, "risk_target": 0.1, "selection_scope": "same development data; exploratory", "eligible_ids": ["odql_7980", "qg_749", "odql_11176", "sfq_20838", "sfq_2187", "bb_8725", "qw_10107", "odql_11899", "sfq_8726", "odql_10108", "odql_3325", "tb_1852", "qg_152", "sfq_4423", "sfq_10592", "sfq_18424", "odql_2963", "sfq_12277", "sfq_17015", "bb_9117", "qz_1730", "sfq_23986", "qf_2590", "jp_2935", "sfq_7702", "sfq_23359", "odql_13834", "qg_774", "odql_7981", "wh_2615", "qw_10089", "qw_4795", "dpql_4823", "qb_1953", "qb_292", "odql_9495", "odql_11469", "bb_3733", "odql_14999", "sfq_1917", "qg_527", "qf_3609", "sfq_22986", "sfq_21768", "qb_2880", "bb_2139", "qw_45", "qw_3576", "sfq_6988", "qb_2169", "qf_387", "sfq_3731", "tb_83", "qg_633", "wh_267", "odql_14829", "sfq_15164", "qw_4450", "qb_4066", "qb_3535", "bt_1378", "qg_504", "bt_61", "bb_7375", "bb_1588", "qb_3775", "sfq_15066", "dpql_782", "odql_1772", "qz_6456", "odql_8729", "qf_2569", "odql_8774", "bt_4062", "wh_4082", "qw_2489", "sfq_24884", "qb_8477", "dpql_1994", "odql_13686", "bb_5892", "qg_2735", "odql_2476", "bt_3047", "odql_8663", "qb_4792", "dpql_3600", "sfq_23513", "bt_4028", "sfq_4588", "dpql_2210", "bb_6465", "qb_38", "qz_4435", "odql_4457", "odql_9352", "sfq_19745", "bb_2691", "odql_6624", "qb_8289", "qw_12650", "bt_1819", "sfq_11945", "qz_2822", "qb_7094", "jp_4067", "sfq_6665", "bt_434", "odql_136", "odql_6494", "qz_1047", "qz_4231", "qw_6139", "bt_3299", "qf_277", "odql_249", "dpql_1447", "sfq_13640", "dpql_580", "qf_2048", "dpql_3467", "qb_3556", "qw_2843", "sfq_19473", "jp_4032", "qw_7594", "sfq_3202", "jp_3453", "qw_11112", "qf_345", "dpql_3635", "jp_2435", "odql_13482", "qw_1298", "sfq_15917", "odql_8493", "qg_357", "sfq_5739", "qg_1580", "odql_5501", "dpql_151", "qg_1800", "bb_7364", "qw_16265", "sfq_6175", "jp_1572", "qw_15027", "qz_6596", "odql_7507", "sfq_17997", "qb_3940", "qw_14565", "odql_7762", "qw_3207", "sfq_2355", "sfq_14806", "sfq_10687", "odql_8306", "sfq_16554", "odql_15094", "bb_6092", "qg_2424", "tb_299", "qb_9584", "sfq_7904", "sfq_10502", "tb_1911", "qg_3056", "bt_1811", "bt_371", "qz_5459", "qb_4163", "qb_8707", "odql_14527", "qg_2316", "dpql_1746", "qb_6635", "odql_2749", "odql_4794", "tc_719", "bt_1604", "qf_2264", "sfq_17840", "qb_2148", "dpql_5194", "tc_1944", "qw_15528", "odql_2965", "sfq_14668", "qw_10673", "dpql_4666", "jp_1502", "sfq_6752"]}, {"condition": "dp_sigma_0p5", "score": "mean_answer_nll", "threshold": 0.36266133189201355, "abstain_all": false, "risk_target": 0.1, "selection_scope": "same development data; exploratory", "eligible_ids": ["odql_7980", "qg_749", "odql_11176", "sfq_20838", "sfq_2187", "bb_8725", "qw_10107", "odql_11899", "sfq_8726", "odql_10108", "odql_3325", "tb_1852", "qg_152", "sfq_4423", "sfq_10592", "sfq_18424", "odql_2963", "sfq_12277", "sfq_17015", "bb_9117", "qz_1730", "sfq_23986", "qf_2590", "jp_2935", "sfq_7702", "sfq_23359", "odql_13834", "qg_774", "odql_7981", "wh_2615", "qw_10089", "qw_4795", "dpql_4823", "qb_1953", "qb_292", "odql_9495", "odql_11469", "bb_3733", "odql_14999", "sfq_1917", "qg_527", "qf_3609", "sfq_22986", "sfq_21768", "qb_2880", "bb_2139", "qw_45", "qw_3576", "sfq_6988", "qb_2169", "qf_387", "sfq_3731", "tb_83", "qg_633", "wh_267", "odql_14829", "sfq_15164", "qw_4450", "qb_4066", "qb_3535", "bt_1378", "qg_504", "bt_61", "bb_7375", "bb_1588", "qb_3775", "sfq_15066", "dpql_782", "odql_1772", "qz_6456", "odql_8729", "qf_2569", "odql_8774", "bt_4062", "wh_4082", "qw_2489", "sfq_24884", "qb_8477", "dpql_1994", "odql_13686", "bb_5892", "qg_2735", "odql_2476", "bt_3047", "odql_8663", "qb_4792", "dpql_3600", "sfq_23513", "bt_4028", "sfq_4588", "dpql_2210", "bb_6465", "qb_38", "qz_4435", "odql_4457", "odql_9352", "sfq_19745", "bb_2691", "odql_6624", "qb_8289", "qw_12650", "bt_1819", "sfq_11945", "qz_2822", "qb_7094", "jp_4067", "sfq_6665", "bt_434", "odql_136", "odql_6494", "qz_1047", "qz_4231", "qw_6139", "bt_3299", "qf_277", "odql_249", "dpql_1447", "sfq_13640", "dpql_580", "qf_2048", "dpql_3467", "qb_3556", "qw_2843", "sfq_19473", "jp_4032", "qw_7594", "sfq_3202", "jp_3453", "qw_11112", "qf_345", "dpql_3635", "jp_2435", "odql_13482", "qw_1298", "sfq_15917", "odql_8493", "qg_357", "sfq_5739", "qg_1580", "odql_5501", "dpql_151", "qg_1800", "bb_7364", "qw_16265", "sfq_6175", "jp_1572", "qw_15027", "qz_6596", "odql_7507", "sfq_17997", "qb_3940", "qw_14565", "odql_7762", "qw_3207", "sfq_2355", "sfq_14806", "sfq_10687", "odql_8306", "sfq_16554", "odql_15094", "bb_6092", "qg_2424", "tb_299", "qb_9584", "sfq_7904", "sfq_10502", "tb_1911", "qg_3056", "bt_1811", "bt_371", "qz_5459", "qb_4163", "qb_8707", "odql_14527", "qg_2316", "dpql_1746", "qb_6635", "odql_2749", "odql_4794", "tc_719", "bt_1604", "qf_2264", "sfq_17840", "qb_2148", "dpql_5194", "tc_1944", "qw_15528", "odql_2965", "sfq_14668", "qw_10673", "dpql_4666", "jp_1502", "sfq_6752"]}, {"condition": "dp_sigma_0p5", "score": "discrete_semantic_entropy", "threshold": 0.6108643020548935, "abstain_all": false, "risk_target": 0.1, "selection_scope": "same development data; exploratory", "eligible_ids": ["odql_7980", "qg_749", "odql_11176", "sfq_20838", "sfq_2187", "bb_8725", "qw_10107", "odql_11899", "sfq_8726", "odql_10108", "odql_3325", "tb_1852", "qg_152", "sfq_4423", "sfq_10592", "sfq_18424", "odql_2963", "sfq_12277", "sfq_17015", "bb_9117", "qz_1730", "sfq_23986", "qf_2590", "jp_2935", "sfq_7702", "sfq_23359", "odql_13834", "qg_774", "odql_7981", "wh_2615", "qw_10089", "qw_4795", "dpql_4823", "qb_1953", "qb_292", "odql_9495", "odql_11469", "bb_3733", "odql_14999", "sfq_1917", "qg_527", "qf_3609", "sfq_22986", "sfq_21768", "qb_2880", "bb_2139", "qw_45", "qw_3576", "sfq_6988", "qb_2169", "qf_387", "sfq_3731", "tb_83", "qg_633", "wh_267", "odql_14829", "sfq_15164", "qw_4450", "qb_4066", "qb_3535", "bt_1378", "qg_504", "bt_61", "bb_7375", "bb_1588", "qb_3775", "sfq_15066", "dpql_782", "odql_1772", "qz_6456", "odql_8729", "qf_2569", "odql_8774", "bt_4062", "wh_4082", "qw_2489", "sfq_24884", "qb_8477", "dpql_1994", "odql_13686", "bb_5892", "qg_2735", "odql_2476", "bt_3047", "odql_8663", "qb_4792", "dpql_3600", "sfq_23513", "bt_4028", "sfq_4588", "dpql_2210", "bb_6465", "qb_38", "qz_4435", "odql_4457", "odql_9352", "sfq_19745", "bb_2691", "odql_6624", "qb_8289", "qw_12650", "bt_1819", "sfq_11945", "qz_2822", "qb_7094", "jp_4067", "sfq_6665", "bt_434", "odql_136", "odql_6494", "qz_1047", "qz_4231", "qw_6139", "bt_3299", "qf_277", "odql_249", "dpql_1447", "sfq_13640", "dpql_580", "qf_2048", "dpql_3467", "qb_3556", "qw_2843", "sfq_19473", "jp_4032", "qw_7594", "sfq_3202", "jp_3453", "qw_11112", "qf_345", "dpql_3635", "jp_2435", "odql_13482", "qw_1298", "sfq_15917", "odql_8493", "qg_357", "sfq_5739", "qg_1580", "odql_5501", "dpql_151", "qg_1800", "bb_7364", "qw_16265", "sfq_6175", "jp_1572", "qw_15027", "qz_6596", "odql_7507", "sfq_17997", "qb_3940", "qw_14565", "odql_7762", "qw_3207", "sfq_2355", "sfq_14806", "sfq_10687", "odql_8306", "sfq_16554", "odql_15094", "bb_6092", "qg_2424", "tb_299", "qb_9584", "sfq_7904", "sfq_10502", "tb_1911", "qg_3056", "bt_1811", "bt_371", "qz_5459", "qb_4163", "qb_8707", "odql_14527", "qg_2316", "dpql_1746", "qb_6635", "odql_2749", "odql_4794", "tc_719", "bt_1604", "qf_2264", "sfq_17840", "qb_2148", "dpql_5194", "tc_1944", "qw_15528", "odql_2965", "sfq_14668", "qw_10673", "dpql_4666", "jp_1502", "sfq_6752"]}], "adapter_file_hashes": {"standard_poisson": "efb1acda87bf677dc764a986aa4ac1444be97e299caed1713309629cdaffb830", "clipping_poisson": "3b8f680bdde96e6f846561878b4cec2dd8e34fade7a68d6004f047f8c70bcdbe", "dp_sigma_0p1": "27fccb501a29731e5294b904df474796c991599f7b63010eff97a2d4ecbce744", "dp_sigma_0p25": "dae3e78e65932e4b490cda399f05c69b503403e00183642d351ab2501c3c2883", "dp_sigma_0p5": "5570fd282f0539c597bf6c344aacdd8e960366b1ad82ea7ad7709af6c58f15a0"}, "development_report": "0aced0d17069", "threshold_file_sha256": "c2d85beb5a03212f12f1acbaeca06bccf4998b5e17e1a1c55a44ed5d468eb8dc"}, "456": {"suite_config_hash": "c62bdd1beb1363f4043fd037efd7062fe1dc54fbe7fa33039a1c1dcff9500345", "thresholds": [{"condition": "standard_poisson", "score": "mean_token_entropy", "threshold": 1.3541622161865234, "abstain_all": false, "risk_target": 0.1, "selection_scope": "same development data; exploratory", "eligible_ids": ["odql_7980", "qg_749", "odql_11176", "sfq_20838", "sfq_2187", "bb_8725", "qw_10107", "odql_11899", "sfq_8726", "odql_10108", "odql_3325", "tb_1852", "qg_152", "sfq_4423", "sfq_10592", "sfq_18424", "odql_2963", "sfq_12277", "sfq_17015", "bb_9117", "qz_1730", "sfq_23986", "qf_2590", "jp_2935", "sfq_7702", "sfq_23359", "odql_13834", "qg_774", "odql_7981", "wh_2615", "qw_10089", "qw_4795", "dpql_4823", "qb_1953", "qb_292", "odql_9495", "odql_11469", "bb_3733", "odql_14999", "sfq_1917", "qg_527", "qf_3609", "sfq_22986", "sfq_21768", "qb_2880", "bb_2139", "qw_45", "qw_3576", "sfq_6988", "qb_2169", "qf_387", "sfq_3731", "tb_83", "qg_633", "wh_267", "odql_14829", "sfq_15164", "qw_4450", "qb_4066", "qb_3535", "bt_1378", "qg_504", "bt_61", "bb_7375", "bb_1588", "qb_3775", "sfq_15066", "dpql_782", "odql_1772", "qz_6456", "odql_8729", "qf_2569", "odql_8774", "bt_4062", "wh_4082", "qw_2489", "sfq_24884", "qb_8477", "dpql_1994", "odql_13686", "bb_5892", "qg_2735", "odql_2476", "bt_3047", "odql_8663", "qb_4792", "dpql_3600", "sfq_23513", "bt_4028", "sfq_4588", "dpql_2210", "bb_6465", "qb_38", "qz_4435", "odql_4457", "odql_9352", "sfq_19745", "bb_2691", "odql_6624", "qb_8289", "qw_12650", "bt_1819", "sfq_11945", "qz_2822", "qb_7094", "jp_4067", "sfq_6665", "bt_434", "odql_136", "odql_6494", "qz_1047", "qz_4231", "qw_6139", "bt_3299", "qf_277", "odql_249", "dpql_1447", "sfq_13640", "dpql_580", "qf_2048", "dpql_3467", "qb_3556", "qw_2843", "sfq_19473", "jp_4032", "qw_7594", "sfq_3202", "jp_3453", "qw_11112", "qf_345", "dpql_3635", "jp_2435", "odql_13482", "qw_1298", "sfq_15917", "odql_8493", "qg_357", "sfq_5739", "qg_1580", "odql_5501", "dpql_151", "qg_1800", "bb_7364", "qw_16265", "sfq_6175", "jp_1572", "qw_15027", "qz_6596", "odql_7507", "sfq_17997", "qb_3940", "qw_14565", "odql_7762", "qw_3207", "sfq_2355", "sfq_14806", "sfq_10687", "odql_8306", "sfq_16554", "odql_15094", "bb_6092", "qg_2424", "tb_299", "qb_9584", "sfq_7904", "sfq_10502", "tb_1911", "qg_3056", "bt_1811", "bt_371", "qz_5459", "qb_4163", "qb_8707", "odql_14527", "qg_2316", "dpql_1746", "qb_6635", "odql_2749", "odql_4794", "tc_719", "bt_1604", "qf_2264", "sfq_17840", "qb_2148", "dpql_5194", "tc_1944", "qw_15528", "odql_2965", "sfq_14668", "qw_10673", "dpql_4666", "jp_1502", "sfq_6752"]}, {"condition": "standard_poisson", "score": "mean_answer_nll", "threshold": 0.5353540182113647, "abstain_all": false, "risk_target": 0.1, "selection_scope": "same development data; exploratory", "eligible_ids": ["odql_7980", "qg_749", "odql_11176", "sfq_20838", "sfq_2187", "bb_8725", "qw_10107", "odql_11899", "sfq_8726", "odql_10108", "odql_3325", "tb_1852", "qg_152", "sfq_4423", "sfq_10592", "sfq_18424", "odql_2963", "sfq_12277", "sfq_17015", "bb_9117", "qz_1730", "sfq_23986", "qf_2590", "jp_2935", "sfq_7702", "sfq_23359", "odql_13834", "qg_774", "odql_7981", "wh_2615", "qw_10089", "qw_4795", "dpql_4823", "qb_1953", "qb_292", "odql_9495", "odql_11469", "bb_3733", "odql_14999", "sfq_1917", "qg_527", "qf_3609", "sfq_22986", "sfq_21768", "qb_2880", "bb_2139", "qw_45", "qw_3576", "sfq_6988", "qb_2169", "qf_387", "sfq_3731", "tb_83", "qg_633", "wh_267", "odql_14829", "sfq_15164", "qw_4450", "qb_4066", "qb_3535", "bt_1378", "qg_504", "bt_61", "bb_7375", "bb_1588", "qb_3775", "sfq_15066", "dpql_782", "odql_1772", "qz_6456", "odql_8729", "qf_2569", "odql_8774", "bt_4062", "wh_4082", "qw_2489", "sfq_24884", "qb_8477", "dpql_1994", "odql_13686", "bb_5892", "qg_2735", "odql_2476", "bt_3047", "odql_8663", "qb_4792", "dpql_3600", "sfq_23513", "bt_4028", "sfq_4588", "dpql_2210", "bb_6465", "qb_38", "qz_4435", "odql_4457", "odql_9352", "sfq_19745", "bb_2691", "odql_6624", "qb_8289", "qw_12650", "bt_1819", "sfq_11945", "qz_2822", "qb_7094", "jp_4067", "sfq_6665", "bt_434", "odql_136", "odql_6494", "qz_1047", "qz_4231", "qw_6139", "bt_3299", "qf_277", "odql_249", "dpql_1447", "sfq_13640", "dpql_580", "qf_2048", "dpql_3467", "qb_3556", "qw_2843", "sfq_19473", "jp_4032", "qw_7594", "sfq_3202", "jp_3453", "qw_11112", "qf_345", "dpql_3635", "jp_2435", "odql_13482", "qw_1298", "sfq_15917", "odql_8493", "qg_357", "sfq_5739", "qg_1580", "odql_5501", "dpql_151", "qg_1800", "bb_7364", "qw_16265", "sfq_6175", "jp_1572", "qw_15027", "qz_6596", "odql_7507", "sfq_17997", "qb_3940", "qw_14565", "odql_7762", "qw_3207", "sfq_2355", "sfq_14806", "sfq_10687", "odql_8306", "sfq_16554", "odql_15094", "bb_6092", "qg_2424", "tb_299", "qb_9584", "sfq_7904", "sfq_10502", "tb_1911", "qg_3056", "bt_1811", "bt_371", "qz_5459", "qb_4163", "qb_8707", "odql_14527", "qg_2316", "dpql_1746", "qb_6635", "odql_2749", "odql_4794", "tc_719", "bt_1604", "qf_2264", "sfq_17840", "qb_2148", "dpql_5194", "tc_1944", "qw_15528", "odql_2965", "sfq_14668", "qw_10673", "dpql_4666", "jp_1502", "sfq_6752"]}, {"condition": "standard_poisson", "score": "discrete_semantic_entropy", "threshold": 0.6931471805599453, "abstain_all": false, "risk_target": 0.1, "selection_scope": "same development data; exploratory", "eligible_ids": ["odql_7980", "qg_749", "odql_11176", "sfq_20838", "sfq_2187", "bb_8725", "qw_10107", "odql_11899", "sfq_8726", "odql_10108", "odql_3325", "tb_1852", "qg_152", "sfq_4423", "sfq_10592", "sfq_18424", "odql_2963", "sfq_12277", "sfq_17015", "bb_9117", "qz_1730", "sfq_23986", "qf_2590", "jp_2935", "sfq_7702", "sfq_23359", "odql_13834", "qg_774", "odql_7981", "wh_2615", "qw_10089", "qw_4795", "dpql_4823", "qb_1953", "qb_292", "odql_9495", "odql_11469", "bb_3733", "odql_14999", "sfq_1917", "qg_527", "qf_3609", "sfq_22986", "sfq_21768", "qb_2880", "bb_2139", "qw_45", "qw_3576", "sfq_6988", "qb_2169", "qf_387", "sfq_3731", "tb_83", "qg_633", "wh_267", "odql_14829", "sfq_15164", "qw_4450", "qb_4066", "qb_3535", "bt_1378", "qg_504", "bt_61", "bb_7375", "bb_1588", "qb_3775", "sfq_15066", "dpql_782", "odql_1772", "qz_6456", "odql_8729", "qf_2569", "odql_8774", "bt_4062", "wh_4082", "qw_2489", "sfq_24884", "qb_8477", "dpql_1994", "odql_13686", "bb_5892", "qg_2735", "odql_2476", "bt_3047", "odql_8663", "qb_4792", "dpql_3600", "sfq_23513", "bt_4028", "sfq_4588", "dpql_2210", "bb_6465", "qb_38", "qz_4435", "odql_4457", "odql_9352", "sfq_19745", "bb_2691", "odql_6624", "qb_8289", "qw_12650", "bt_1819", "sfq_11945", "qz_2822", "qb_7094", "jp_4067", "sfq_6665", "bt_434", "odql_136", "odql_6494", "qz_1047", "qz_4231", "qw_6139", "bt_3299", "qf_277", "odql_249", "dpql_1447", "sfq_13640", "dpql_580", "qf_2048", "dpql_3467", "qb_3556", "qw_2843", "sfq_19473", "jp_4032", "qw_7594", "sfq_3202", "jp_3453", "qw_11112", "qf_345", "dpql_3635", "jp_2435", "odql_13482", "qw_1298", "sfq_15917", "odql_8493", "qg_357", "sfq_5739", "qg_1580", "odql_5501", "dpql_151", "qg_1800", "bb_7364", "qw_16265", "sfq_6175", "jp_1572", "qw_15027", "qz_6596", "odql_7507", "sfq_17997", "qb_3940", "qw_14565", "odql_7762", "qw_3207", "sfq_2355", "sfq_14806", "sfq_10687", "odql_8306", "sfq_16554", "odql_15094", "bb_6092", "qg_2424", "tb_299", "qb_9584", "sfq_7904", "sfq_10502", "tb_1911", "qg_3056", "bt_1811", "bt_371", "qz_5459", "qb_4163", "qb_8707", "odql_14527", "qg_2316", "dpql_1746", "qb_6635", "odql_2749", "odql_4794", "tc_719", "bt_1604", "qf_2264", "sfq_17840", "qb_2148", "dpql_5194", "tc_1944", "qw_15528", "odql_2965", "sfq_14668", "qw_10673", "dpql_4666", "jp_1502", "sfq_6752"]}, {"condition": "clipping_poisson", "score": "mean_token_entropy", "threshold": 1.0519076585769653, "abstain_all": false, "risk_target": 0.1, "selection_scope": "same development data; exploratory", "eligible_ids": ["odql_7980", "qg_749", "odql_11176", "sfq_20838", "sfq_2187", "bb_8725", "qw_10107", "odql_11899", "sfq_8726", "odql_10108", "odql_3325", "tb_1852", "qg_152", "sfq_4423", "sfq_10592", "sfq_18424", "odql_2963", "sfq_12277", "sfq_17015", "bb_9117", "qz_1730", "sfq_23986", "qf_2590", "jp_2935", "sfq_7702", "sfq_23359", "odql_13834", "qg_774", "odql_7981", "wh_2615", "qw_10089", "qw_4795", "dpql_4823", "qb_1953", "qb_292", "odql_9495", "odql_11469", "bb_3733", "odql_14999", "sfq_1917", "qg_527", "qf_3609", "sfq_22986", "sfq_21768", "qb_2880", "bb_2139", "qw_45", "qw_3576", "sfq_6988", "qb_2169", "qf_387", "sfq_3731", "tb_83", "qg_633", "wh_267", "odql_14829", "sfq_15164", "qw_4450", "qb_4066", "qb_3535", "bt_1378", "qg_504", "bt_61", "bb_7375", "bb_1588", "qb_3775", "sfq_15066", "dpql_782", "odql_1772", "qz_6456", "odql_8729", "qf_2569", "odql_8774", "bt_4062", "wh_4082", "qw_2489", "sfq_24884", "qb_8477", "dpql_1994", "odql_13686", "bb_5892", "qg_2735", "odql_2476", "bt_3047", "odql_8663", "qb_4792", "dpql_3600", "sfq_23513", "bt_4028", "sfq_4588", "dpql_2210", "bb_6465", "qb_38", "qz_4435", "odql_4457", "odql_9352", "sfq_19745", "bb_2691", "odql_6624", "qb_8289", "qw_12650", "bt_1819", "sfq_11945", "qz_2822", "qb_7094", "jp_4067", "sfq_6665", "bt_434", "odql_136", "odql_6494", "qz_1047", "qz_4231", "qw_6139", "bt_3299", "qf_277", "odql_249", "dpql_1447", "sfq_13640", "dpql_580", "qf_2048", "dpql_3467", "qb_3556", "qw_2843", "sfq_19473", "jp_4032", "qw_7594", "sfq_3202", "jp_3453", "qw_11112", "qf_345", "dpql_3635", "jp_2435", "odql_13482", "qw_1298", "sfq_15917", "odql_8493", "qg_357", "sfq_5739", "qg_1580", "odql_5501", "dpql_151", "qg_1800", "bb_7364", "qw_16265", "sfq_6175", "jp_1572", "qw_15027", "qz_6596", "odql_7507", "sfq_17997", "qb_3940", "qw_14565", "odql_7762", "qw_3207", "sfq_2355", "sfq_14806", "sfq_10687", "odql_8306", "sfq_16554", "odql_15094", "bb_6092", "qg_2424", "tb_299", "qb_9584", "sfq_7904", "sfq_10502", "tb_1911", "qg_3056", "bt_1811", "bt_371", "qz_5459", "qb_4163", "qb_8707", "odql_14527", "qg_2316", "dpql_1746", "qb_6635", "odql_2749", "odql_4794", "tc_719", "bt_1604", "qf_2264", "sfq_17840", "qb_2148", "dpql_5194", "tc_1944", "qw_15528", "odql_2965", "sfq_14668", "qw_10673", "dpql_4666", "jp_1502", "sfq_6752"]}, {"condition": "clipping_poisson", "score": "mean_answer_nll", "threshold": 0.41581782698631287, "abstain_all": false, "risk_target": 0.1, "selection_scope": "same development data; exploratory", "eligible_ids": ["odql_7980", "qg_749", "odql_11176", "sfq_20838", "sfq_2187", "bb_8725", "qw_10107", "odql_11899", "sfq_8726", "odql_10108", "odql_3325", "tb_1852", "qg_152", "sfq_4423", "sfq_10592", "sfq_18424", "odql_2963", "sfq_12277", "sfq_17015", "bb_9117", "qz_1730", "sfq_23986", "qf_2590", "jp_2935", "sfq_7702", "sfq_23359", "odql_13834", "qg_774", "odql_7981", "wh_2615", "qw_10089", "qw_4795", "dpql_4823", "qb_1953", "qb_292", "odql_9495", "odql_11469", "bb_3733", "odql_14999", "sfq_1917", "qg_527", "qf_3609", "sfq_22986", "sfq_21768", "qb_2880", "bb_2139", "qw_45", "qw_3576", "sfq_6988", "qb_2169", "qf_387", "sfq_3731", "tb_83", "qg_633", "wh_267", "odql_14829", "sfq_15164", "qw_4450", "qb_4066", "qb_3535", "bt_1378", "qg_504", "bt_61", "bb_7375", "bb_1588", "qb_3775", "sfq_15066", "dpql_782", "odql_1772", "qz_6456", "odql_8729", "qf_2569", "odql_8774", "bt_4062", "wh_4082", "qw_2489", "sfq_24884", "qb_8477", "dpql_1994", "odql_13686", "bb_5892", "qg_2735", "odql_2476", "bt_3047", "odql_8663", "qb_4792", "dpql_3600", "sfq_23513", "bt_4028", "sfq_4588", "dpql_2210", "bb_6465", "qb_38", "qz_4435", "odql_4457", "odql_9352", "sfq_19745", "bb_2691", "odql_6624", "qb_8289", "qw_12650", "bt_1819", "sfq_11945", "qz_2822", "qb_7094", "jp_4067", "sfq_6665", "bt_434", "odql_136", "odql_6494", "qz_1047", "qz_4231", "qw_6139", "bt_3299", "qf_277", "odql_249", "dpql_1447", "sfq_13640", "dpql_580", "qf_2048", "dpql_3467", "qb_3556", "qw_2843", "sfq_19473", "jp_4032", "qw_7594", "sfq_3202", "jp_3453", "qw_11112", "qf_345", "dpql_3635", "jp_2435", "odql_13482", "qw_1298", "sfq_15917", "odql_8493", "qg_357", "sfq_5739", "qg_1580", "odql_5501", "dpql_151", "qg_1800", "bb_7364", "qw_16265", "sfq_6175", "jp_1572", "qw_15027", "qz_6596", "odql_7507", "sfq_17997", "qb_3940", "qw_14565", "odql_7762", "qw_3207", "sfq_2355", "sfq_14806", "sfq_10687", "odql_8306", "sfq_16554", "odql_15094", "bb_6092", "qg_2424", "tb_299", "qb_9584", "sfq_7904", "sfq_10502", "tb_1911", "qg_3056", "bt_1811", "bt_371", "qz_5459", "qb_4163", "qb_8707", "odql_14527", "qg_2316", "dpql_1746", "qb_6635", "odql_2749", "odql_4794", "tc_719", "bt_1604", "qf_2264", "sfq_17840", "qb_2148", "dpql_5194", "tc_1944", "qw_15528", "odql_2965", "sfq_14668", "qw_10673", "dpql_4666", "jp_1502", "sfq_6752"]}, {"condition": "clipping_poisson", "score": "discrete_semantic_entropy", "threshold": 0.639031859650177, "abstain_all": false, "risk_target": 0.1, "selection_scope": "same development data; exploratory", "eligible_ids": ["odql_7980", "qg_749", "odql_11176", "sfq_20838", "sfq_2187", "bb_8725", "qw_10107", "odql_11899", "sfq_8726", "odql_10108", "odql_3325", "tb_1852", "qg_152", "sfq_4423", "sfq_10592", "sfq_18424", "odql_2963", "sfq_12277", "sfq_17015", "bb_9117", "qz_1730", "sfq_23986", "qf_2590", "jp_2935", "sfq_7702", "sfq_23359", "odql_13834", "qg_774", "odql_7981", "wh_2615", "qw_10089", "qw_4795", "dpql_4823", "qb_1953", "qb_292", "odql_9495", "odql_11469", "bb_3733", "odql_14999", "sfq_1917", "qg_527", "qf_3609", "sfq_22986", "sfq_21768", "qb_2880", "bb_2139", "qw_45", "qw_3576", "sfq_6988", "qb_2169", "qf_387", "sfq_3731", "tb_83", "qg_633", "wh_267", "odql_14829", "sfq_15164", "qw_4450", "qb_4066", "qb_3535", "bt_1378", "qg_504", "bt_61", "bb_7375", "bb_1588", "qb_3775", "sfq_15066", "dpql_782", "odql_1772", "qz_6456", "odql_8729", "qf_2569", "odql_8774", "bt_4062", "wh_4082", "qw_2489", "sfq_24884", "qb_8477", "dpql_1994", "odql_13686", "bb_5892", "qg_2735", "odql_2476", "bt_3047", "odql_8663", "qb_4792", "dpql_3600", "sfq_23513", "bt_4028", "sfq_4588", "dpql_2210", "bb_6465", "qb_38", "qz_4435", "odql_4457", "odql_9352", "sfq_19745", "bb_2691", "odql_6624", "qb_8289", "qw_12650", "bt_1819", "sfq_11945", "qz_2822", "qb_7094", "jp_4067", "sfq_6665", "bt_434", "odql_136", "odql_6494", "qz_1047", "qz_4231", "qw_6139", "bt_3299", "qf_277", "odql_249", "dpql_1447", "sfq_13640", "dpql_580", "qf_2048", "dpql_3467", "qb_3556", "qw_2843", "sfq_19473", "jp_4032", "qw_7594", "sfq_3202", "jp_3453", "qw_11112", "qf_345", "dpql_3635", "jp_2435", "odql_13482", "qw_1298", "sfq_15917", "odql_8493", "qg_357", "sfq_5739", "qg_1580", "odql_5501", "dpql_151", "qg_1800", "bb_7364", "qw_16265", "sfq_6175", "jp_1572", "qw_15027", "qz_6596", "odql_7507", "sfq_17997", "qb_3940", "qw_14565", "odql_7762", "qw_3207", "sfq_2355", "sfq_14806", "sfq_10687", "odql_8306", "sfq_16554", "odql_15094", "bb_6092", "qg_2424", "tb_299", "qb_9584", "sfq_7904", "sfq_10502", "tb_1911", "qg_3056", "bt_1811", "bt_371", "qz_5459", "qb_4163", "qb_8707", "odql_14527", "qg_2316", "dpql_1746", "qb_6635", "odql_2749", "odql_4794", "tc_719", "bt_1604", "qf_2264", "sfq_17840", "qb_2148", "dpql_5194", "tc_1944", "qw_15528", "odql_2965", "sfq_14668", "qw_10673", "dpql_4666", "jp_1502", "sfq_6752"]}, {"condition": "dp_sigma_0p1", "score": "mean_token_entropy", "threshold": 1.3046149015426636, "abstain_all": false, "risk_target": 0.1, "selection_scope": "same development data; exploratory", "eligible_ids": ["odql_7980", "qg_749", "odql_11176", "sfq_20838", "sfq_2187", "bb_8725", "qw_10107", "odql_11899", "sfq_8726", "odql_10108", "odql_3325", "tb_1852", "qg_152", "sfq_4423", "sfq_10592", "sfq_18424", "odql_2963", "sfq_12277", "sfq_17015", "bb_9117", "qz_1730", "sfq_23986", "qf_2590", "jp_2935", "sfq_7702", "sfq_23359", "odql_13834", "qg_774", "odql_7981", "wh_2615", "qw_10089", "qw_4795", "dpql_4823", "qb_1953", "qb_292", "odql_9495", "odql_11469", "bb_3733", "odql_14999", "sfq_1917", "qg_527", "qf_3609", "sfq_22986", "sfq_21768", "qb_2880", "bb_2139", "qw_45", "qw_3576", "sfq_6988", "qb_2169", "qf_387", "sfq_3731", "tb_83", "qg_633", "wh_267", "odql_14829", "sfq_15164", "qw_4450", "qb_4066", "qb_3535", "bt_1378", "qg_504", "bt_61", "bb_7375", "bb_1588", "qb_3775", "sfq_15066", "dpql_782", "odql_1772", "qz_6456", "odql_8729", "qf_2569", "odql_8774", "bt_4062", "wh_4082", "qw_2489", "sfq_24884", "qb_8477", "dpql_1994", "odql_13686", "bb_5892", "qg_2735", "odql_2476", "bt_3047", "odql_8663", "qb_4792", "dpql_3600", "sfq_23513", "bt_4028", "sfq_4588", "dpql_2210", "bb_6465", "qb_38", "qz_4435", "odql_4457", "odql_9352", "sfq_19745", "bb_2691", "odql_6624", "qb_8289", "qw_12650", "bt_1819", "sfq_11945", "qz_2822", "qb_7094", "jp_4067", "sfq_6665", "bt_434", "odql_136", "odql_6494", "qz_1047", "qz_4231", "qw_6139", "bt_3299", "qf_277", "odql_249", "dpql_1447", "sfq_13640", "dpql_580", "qf_2048", "dpql_3467", "qb_3556", "qw_2843", "sfq_19473", "jp_4032", "qw_7594", "sfq_3202", "jp_3453", "qw_11112", "qf_345", "dpql_3635", "jp_2435", "odql_13482", "qw_1298", "sfq_15917", "odql_8493", "qg_357", "sfq_5739", "qg_1580", "odql_5501", "dpql_151", "qg_1800", "bb_7364", "qw_16265", "sfq_6175", "jp_1572", "qw_15027", "qz_6596", "odql_7507", "sfq_17997", "qb_3940", "qw_14565", "odql_7762", "qw_3207", "sfq_2355", "sfq_14806", "sfq_10687", "odql_8306", "sfq_16554", "odql_15094", "bb_6092", "qg_2424", "tb_299", "qb_9584", "sfq_7904", "sfq_10502", "tb_1911", "qg_3056", "bt_1811", "bt_371", "qz_5459", "qb_4163", "qb_8707", "odql_14527", "qg_2316", "dpql_1746", "qb_6635", "odql_2749", "odql_4794", "tc_719", "bt_1604", "qf_2264", "sfq_17840", "qb_2148", "dpql_5194", "tc_1944", "qw_15528", "odql_2965", "sfq_14668", "qw_10673", "dpql_4666", "jp_1502", "sfq_6752"]}, {"condition": "dp_sigma_0p1", "score": "mean_answer_nll", "threshold": 0.5149239897727966, "abstain_all": false, "risk_target": 0.1, "selection_scope": "same development data; exploratory", "eligible_ids": ["odql_7980", "qg_749", "odql_11176", "sfq_20838", "sfq_2187", "bb_8725", "qw_10107", "odql_11899", "sfq_8726", "odql_10108", "odql_3325", "tb_1852", "qg_152", "sfq_4423", "sfq_10592", "sfq_18424", "odql_2963", "sfq_12277", "sfq_17015", "bb_9117", "qz_1730", "sfq_23986", "qf_2590", "jp_2935", "sfq_7702", "sfq_23359", "odql_13834", "qg_774", "odql_7981", "wh_2615", "qw_10089", "qw_4795", "dpql_4823", "qb_1953", "qb_292", "odql_9495", "odql_11469", "bb_3733", "odql_14999", "sfq_1917", "qg_527", "qf_3609", "sfq_22986", "sfq_21768", "qb_2880", "bb_2139", "qw_45", "qw_3576", "sfq_6988", "qb_2169", "qf_387", "sfq_3731", "tb_83", "qg_633", "wh_267", "odql_14829", "sfq_15164", "qw_4450", "qb_4066", "qb_3535", "bt_1378", "qg_504", "bt_61", "bb_7375", "bb_1588", "qb_3775", "sfq_15066", "dpql_782", "odql_1772", "qz_6456", "odql_8729", "qf_2569", "odql_8774", "bt_4062", "wh_4082", "qw_2489", "sfq_24884", "qb_8477", "dpql_1994", "odql_13686", "bb_5892", "qg_2735", "odql_2476", "bt_3047", "odql_8663", "qb_4792", "dpql_3600", "sfq_23513", "bt_4028", "sfq_4588", "dpql_2210", "bb_6465", "qb_38", "qz_4435", "odql_4457", "odql_9352", "sfq_19745", "bb_2691", "odql_6624", "qb_8289", "qw_12650", "bt_1819", "sfq_11945", "qz_2822", "qb_7094", "jp_4067", "sfq_6665", "bt_434", "odql_136", "odql_6494", "qz_1047", "qz_4231", "qw_6139", "bt_3299", "qf_277", "odql_249", "dpql_1447", "sfq_13640", "dpql_580", "qf_2048", "dpql_3467", "qb_3556", "qw_2843", "sfq_19473", "jp_4032", "qw_7594", "sfq_3202", "jp_3453", "qw_11112", "qf_345", "dpql_3635", "jp_2435", "odql_13482", "qw_1298", "sfq_15917", "odql_8493", "qg_357", "sfq_5739", "qg_1580", "odql_5501", "dpql_151", "qg_1800", "bb_7364", "qw_16265", "sfq_6175", "jp_1572", "qw_15027", "qz_6596", "odql_7507", "sfq_17997", "qb_3940", "qw_14565", "odql_7762", "qw_3207", "sfq_2355", "sfq_14806", "sfq_10687", "odql_8306", "sfq_16554", "odql_15094", "bb_6092", "qg_2424", "tb_299", "qb_9584", "sfq_7904", "sfq_10502", "tb_1911", "qg_3056", "bt_1811", "bt_371", "qz_5459", "qb_4163", "qb_8707", "odql_14527", "qg_2316", "dpql_1746", "qb_6635", "odql_2749", "odql_4794", "tc_719", "bt_1604", "qf_2264", "sfq_17840", "qb_2148", "dpql_5194", "tc_1944", "qw_15528", "odql_2965", "sfq_14668", "qw_10673", "dpql_4666", "jp_1502", "sfq_6752"]}, {"condition": "dp_sigma_0p1", "score": "discrete_semantic_entropy", "threshold": 0.6931471805599453, "abstain_all": false, "risk_target": 0.1, "selection_scope": "same development data; exploratory", "eligible_ids": ["odql_7980", "qg_749", "odql_11176", "sfq_20838", "sfq_2187", "bb_8725", "qw_10107", "odql_11899", "sfq_8726", "odql_10108", "odql_3325", "tb_1852", "qg_152", "sfq_4423", "sfq_10592", "sfq_18424", "odql_2963", "sfq_12277", "sfq_17015", "bb_9117", "qz_1730", "sfq_23986", "qf_2590", "jp_2935", "sfq_7702", "sfq_23359", "odql_13834", "qg_774", "odql_7981", "wh_2615", "qw_10089", "qw_4795", "dpql_4823", "qb_1953", "qb_292", "odql_9495", "odql_11469", "bb_3733", "odql_14999", "sfq_1917", "qg_527", "qf_3609", "sfq_22986", "sfq_21768", "qb_2880", "bb_2139", "qw_45", "qw_3576", "sfq_6988", "qb_2169", "qf_387", "sfq_3731", "tb_83", "qg_633", "wh_267", "odql_14829", "sfq_15164", "qw_4450", "qb_4066", "qb_3535", "bt_1378", "qg_504", "bt_61", "bb_7375", "bb_1588", "qb_3775", "sfq_15066", "dpql_782", "odql_1772", "qz_6456", "odql_8729", "qf_2569", "odql_8774", "bt_4062", "wh_4082", "qw_2489", "sfq_24884", "qb_8477", "dpql_1994", "odql_13686", "bb_5892", "qg_2735", "odql_2476", "bt_3047", "odql_8663", "qb_4792", "dpql_3600", "sfq_23513", "bt_4028", "sfq_4588", "dpql_2210", "bb_6465", "qb_38", "qz_4435", "odql_4457", "odql_9352", "sfq_19745", "bb_2691", "odql_6624", "qb_8289", "qw_12650", "bt_1819", "sfq_11945", "qz_2822", "qb_7094", "jp_4067", "sfq_6665", "bt_434", "odql_136", "odql_6494", "qz_1047", "qz_4231", "qw_6139", "bt_3299", "qf_277", "odql_249", "dpql_1447", "sfq_13640", "dpql_580", "qf_2048", "dpql_3467", "qb_3556", "qw_2843", "sfq_19473", "jp_4032", "qw_7594", "sfq_3202", "jp_3453", "qw_11112", "qf_345", "dpql_3635", "jp_2435", "odql_13482", "qw_1298", "sfq_15917", "odql_8493", "qg_357", "sfq_5739", "qg_1580", "odql_5501", "dpql_151", "qg_1800", "bb_7364", "qw_16265", "sfq_6175", "jp_1572", "qw_15027", "qz_6596", "odql_7507", "sfq_17997", "qb_3940", "qw_14565", "odql_7762", "qw_3207", "sfq_2355", "sfq_14806", "sfq_10687", "odql_8306", "sfq_16554", "odql_15094", "bb_6092", "qg_2424", "tb_299", "qb_9584", "sfq_7904", "sfq_10502", "tb_1911", "qg_3056", "bt_1811", "bt_371", "qz_5459", "qb_4163", "qb_8707", "odql_14527", "qg_2316", "dpql_1746", "qb_6635", "odql_2749", "odql_4794", "tc_719", "bt_1604", "qf_2264", "sfq_17840", "qb_2148", "dpql_5194", "tc_1944", "qw_15528", "odql_2965", "sfq_14668", "qw_10673", "dpql_4666", "jp_1502", "sfq_6752"]}, {"condition": "dp_sigma_0p25", "score": "mean_token_entropy", "threshold": 1.0136948823928833, "abstain_all": false, "risk_target": 0.1, "selection_scope": "same development data; exploratory", "eligible_ids": ["odql_7980", "qg_749", "odql_11176", "sfq_20838", "sfq_2187", "bb_8725", "qw_10107", "odql_11899", "sfq_8726", "odql_10108", "odql_3325", "tb_1852", "qg_152", "sfq_4423", "sfq_10592", "sfq_18424", "odql_2963", "sfq_12277", "sfq_17015", "bb_9117", "qz_1730", "sfq_23986", "qf_2590", "jp_2935", "sfq_7702", "sfq_23359", "odql_13834", "qg_774", "odql_7981", "wh_2615", "qw_10089", "qw_4795", "dpql_4823", "qb_1953", "qb_292", "odql_9495", "odql_11469", "bb_3733", "odql_14999", "sfq_1917", "qg_527", "qf_3609", "sfq_22986", "sfq_21768", "qb_2880", "bb_2139", "qw_45", "qw_3576", "sfq_6988", "qb_2169", "qf_387", "sfq_3731", "tb_83", "qg_633", "wh_267", "odql_14829", "sfq_15164", "qw_4450", "qb_4066", "qb_3535", "bt_1378", "qg_504", "bt_61", "bb_7375", "bb_1588", "qb_3775", "sfq_15066", "dpql_782", "odql_1772", "qz_6456", "odql_8729", "qf_2569", "odql_8774", "bt_4062", "wh_4082", "qw_2489", "sfq_24884", "qb_8477", "dpql_1994", "odql_13686", "bb_5892", "qg_2735", "odql_2476", "bt_3047", "odql_8663", "qb_4792", "dpql_3600", "sfq_23513", "bt_4028", "sfq_4588", "dpql_2210", "bb_6465", "qb_38", "qz_4435", "odql_4457", "odql_9352", "sfq_19745", "bb_2691", "odql_6624", "qb_8289", "qw_12650", "bt_1819", "sfq_11945", "qz_2822", "qb_7094", "jp_4067", "sfq_6665", "bt_434", "odql_136", "odql_6494", "qz_1047", "qz_4231", "qw_6139", "bt_3299", "qf_277", "odql_249", "dpql_1447", "sfq_13640", "dpql_580", "qf_2048", "dpql_3467", "qb_3556", "qw_2843", "sfq_19473", "jp_4032", "qw_7594", "sfq_3202", "jp_3453", "qw_11112", "qf_345", "dpql_3635", "jp_2435", "odql_13482", "qw_1298", "sfq_15917", "odql_8493", "qg_357", "sfq_5739", "qg_1580", "odql_5501", "dpql_151", "qg_1800", "bb_7364", "qw_16265", "sfq_6175", "jp_1572", "qw_15027", "qz_6596", "odql_7507", "sfq_17997", "qb_3940", "qw_14565", "odql_7762", "qw_3207", "sfq_2355", "sfq_14806", "sfq_10687", "odql_8306", "sfq_16554", "odql_15094", "bb_6092", "qg_2424", "tb_299", "qb_9584", "sfq_7904", "sfq_10502", "tb_1911", "qg_3056", "bt_1811", "bt_371", "qz_5459", "qb_4163", "qb_8707", "odql_14527", "qg_2316", "dpql_1746", "qb_6635", "odql_2749", "odql_4794", "tc_719", "bt_1604", "qf_2264", "sfq_17840", "qb_2148", "dpql_5194", "tc_1944", "qw_15528", "odql_2965", "sfq_14668", "qw_10673", "dpql_4666", "jp_1502", "sfq_6752"]}, {"condition": "dp_sigma_0p25", "score": "mean_answer_nll", "threshold": 0.44798046350479126, "abstain_all": false, "risk_target": 0.1, "selection_scope": "same development data; exploratory", "eligible_ids": ["odql_7980", "qg_749", "odql_11176", "sfq_20838", "sfq_2187", "bb_8725", "qw_10107", "odql_11899", "sfq_8726", "odql_10108", "odql_3325", "tb_1852", "qg_152", "sfq_4423", "sfq_10592", "sfq_18424", "odql_2963", "sfq_12277", "sfq_17015", "bb_9117", "qz_1730", "sfq_23986", "qf_2590", "jp_2935", "sfq_7702", "sfq_23359", "odql_13834", "qg_774", "odql_7981", "wh_2615", "qw_10089", "qw_4795", "dpql_4823", "qb_1953", "qb_292", "odql_9495", "odql_11469", "bb_3733", "odql_14999", "sfq_1917", "qg_527", "qf_3609", "sfq_22986", "sfq_21768", "qb_2880", "bb_2139", "qw_45", "qw_3576", "sfq_6988", "qb_2169", "qf_387", "sfq_3731", "tb_83", "qg_633", "wh_267", "odql_14829", "sfq_15164", "qw_4450", "qb_4066", "qb_3535", "bt_1378", "qg_504", "bt_61", "bb_7375", "bb_1588", "qb_3775", "sfq_15066", "dpql_782", "odql_1772", "qz_6456", "odql_8729", "qf_2569", "odql_8774", "bt_4062", "wh_4082", "qw_2489", "sfq_24884", "qb_8477", "dpql_1994", "odql_13686", "bb_5892", "qg_2735", "odql_2476", "bt_3047", "odql_8663", "qb_4792", "dpql_3600", "sfq_23513", "bt_4028", "sfq_4588", "dpql_2210", "bb_6465", "qb_38", "qz_4435", "odql_4457", "odql_9352", "sfq_19745", "bb_2691", "odql_6624", "qb_8289", "qw_12650", "bt_1819", "sfq_11945", "qz_2822", "qb_7094", "jp_4067", "sfq_6665", "bt_434", "odql_136", "odql_6494", "qz_1047", "qz_4231", "qw_6139", "bt_3299", "qf_277", "odql_249", "dpql_1447", "sfq_13640", "dpql_580", "qf_2048", "dpql_3467", "qb_3556", "qw_2843", "sfq_19473", "jp_4032", "qw_7594", "sfq_3202", "jp_3453", "qw_11112", "qf_345", "dpql_3635", "jp_2435", "odql_13482", "qw_1298", "sfq_15917", "odql_8493", "qg_357", "sfq_5739", "qg_1580", "odql_5501", "dpql_151", "qg_1800", "bb_7364", "qw_16265", "sfq_6175", "jp_1572", "qw_15027", "qz_6596", "odql_7507", "sfq_17997", "qb_3940", "qw_14565", "odql_7762", "qw_3207", "sfq_2355", "sfq_14806", "sfq_10687", "odql_8306", "sfq_16554", "odql_15094", "bb_6092", "qg_2424", "tb_299", "qb_9584", "sfq_7904", "sfq_10502", "tb_1911", "qg_3056", "bt_1811", "bt_371", "qz_5459", "qb_4163", "qb_8707", "odql_14527", "qg_2316", "dpql_1746", "qb_6635", "odql_2749", "odql_4794", "tc_719", "bt_1604", "qf_2264", "sfq_17840", "qb_2148", "dpql_5194", "tc_1944", "qw_15528", "odql_2965", "sfq_14668", "qw_10673", "dpql_4666", "jp_1502", "sfq_6752"]}, {"condition": "dp_sigma_0p25", "score": "discrete_semantic_entropy", "threshold": 0.6730116670092565, "abstain_all": false, "risk_target": 0.1, "selection_scope": "same development data; exploratory", "eligible_ids": ["odql_7980", "qg_749", "odql_11176", "sfq_20838", "sfq_2187", "bb_8725", "qw_10107", "odql_11899", "sfq_8726", "odql_10108", "odql_3325", "tb_1852", "qg_152", "sfq_4423", "sfq_10592", "sfq_18424", "odql_2963", "sfq_12277", "sfq_17015", "bb_9117", "qz_1730", "sfq_23986", "qf_2590", "jp_2935", "sfq_7702", "sfq_23359", "odql_13834", "qg_774", "odql_7981", "wh_2615", "qw_10089", "qw_4795", "dpql_4823", "qb_1953", "qb_292", "odql_9495", "odql_11469", "bb_3733", "odql_14999", "sfq_1917", "qg_527", "qf_3609", "sfq_22986", "sfq_21768", "qb_2880", "bb_2139", "qw_45", "qw_3576", "sfq_6988", "qb_2169", "qf_387", "sfq_3731", "tb_83", "qg_633", "wh_267", "odql_14829", "sfq_15164", "qw_4450", "qb_4066", "qb_3535", "bt_1378", "qg_504", "bt_61", "bb_7375", "bb_1588", "qb_3775", "sfq_15066", "dpql_782", "odql_1772", "qz_6456", "odql_8729", "qf_2569", "odql_8774", "bt_4062", "wh_4082", "qw_2489", "sfq_24884", "qb_8477", "dpql_1994", "odql_13686", "bb_5892", "qg_2735", "odql_2476", "bt_3047", "odql_8663", "qb_4792", "dpql_3600", "sfq_23513", "bt_4028", "sfq_4588", "dpql_2210", "bb_6465", "qb_38", "qz_4435", "odql_4457", "odql_9352", "sfq_19745", "bb_2691", "odql_6624", "qb_8289", "qw_12650", "bt_1819", "sfq_11945", "qz_2822", "qb_7094", "jp_4067", "sfq_6665", "bt_434", "odql_136", "odql_6494", "qz_1047", "qz_4231", "qw_6139", "bt_3299", "qf_277", "odql_249", "dpql_1447", "sfq_13640", "dpql_580", "qf_2048", "dpql_3467", "qb_3556", "qw_2843", "sfq_19473", "jp_4032", "qw_7594", "sfq_3202", "jp_3453", "qw_11112", "qf_345", "dpql_3635", "jp_2435", "odql_13482", "qw_1298", "sfq_15917", "odql_8493", "qg_357", "sfq_5739", "qg_1580", "odql_5501", "dpql_151", "qg_1800", "bb_7364", "qw_16265", "sfq_6175", "jp_1572", "qw_15027", "qz_6596", "odql_7507", "sfq_17997", "qb_3940", "qw_14565", "odql_7762", "qw_3207", "sfq_2355", "sfq_14806", "sfq_10687", "odql_8306", "sfq_16554", "odql_15094", "bb_6092", "qg_2424", "tb_299", "qb_9584", "sfq_7904", "sfq_10502", "tb_1911", "qg_3056", "bt_1811", "bt_371", "qz_5459", "qb_4163", "qb_8707", "odql_14527", "qg_2316", "dpql_1746", "qb_6635", "odql_2749", "odql_4794", "tc_719", "bt_1604", "qf_2264", "sfq_17840", "qb_2148", "dpql_5194", "tc_1944", "qw_15528", "odql_2965", "sfq_14668", "qw_10673", "dpql_4666", "jp_1502", "sfq_6752"]}, {"condition": "dp_sigma_0p5", "score": "mean_token_entropy", "threshold": 0.9747848510742188, "abstain_all": false, "risk_target": 0.1, "selection_scope": "same development data; exploratory", "eligible_ids": ["odql_7980", "qg_749", "odql_11176", "sfq_20838", "sfq_2187", "bb_8725", "qw_10107", "odql_11899", "sfq_8726", "odql_10108", "odql_3325", "tb_1852", "qg_152", "sfq_4423", "sfq_10592", "sfq_18424", "odql_2963", "sfq_12277", "sfq_17015", "bb_9117", "qz_1730", "sfq_23986", "qf_2590", "jp_2935", "sfq_7702", "sfq_23359", "odql_13834", "qg_774", "odql_7981", "wh_2615", "qw_10089", "qw_4795", "dpql_4823", "qb_1953", "qb_292", "odql_9495", "odql_11469", "bb_3733", "odql_14999", "sfq_1917", "qg_527", "qf_3609", "sfq_22986", "sfq_21768", "qb_2880", "bb_2139", "qw_45", "qw_3576", "sfq_6988", "qb_2169", "qf_387", "sfq_3731", "tb_83", "qg_633", "wh_267", "odql_14829", "sfq_15164", "qw_4450", "qb_4066", "qb_3535", "bt_1378", "qg_504", "bt_61", "bb_7375", "bb_1588", "qb_3775", "sfq_15066", "dpql_782", "odql_1772", "qz_6456", "odql_8729", "qf_2569", "odql_8774", "bt_4062", "wh_4082", "qw_2489", "sfq_24884", "qb_8477", "dpql_1994", "odql_13686", "bb_5892", "qg_2735", "odql_2476", "bt_3047", "odql_8663", "qb_4792", "dpql_3600", "sfq_23513", "bt_4028", "sfq_4588", "dpql_2210", "bb_6465", "qb_38", "qz_4435", "odql_4457", "odql_9352", "sfq_19745", "bb_2691", "odql_6624", "qb_8289", "qw_12650", "bt_1819", "sfq_11945", "qz_2822", "qb_7094", "jp_4067", "sfq_6665", "bt_434", "odql_136", "odql_6494", "qz_1047", "qz_4231", "qw_6139", "bt_3299", "qf_277", "odql_249", "dpql_1447", "sfq_13640", "dpql_580", "qf_2048", "dpql_3467", "qb_3556", "qw_2843", "sfq_19473", "jp_4032", "qw_7594", "sfq_3202", "jp_3453", "qw_11112", "qf_345", "dpql_3635", "jp_2435", "odql_13482", "qw_1298", "sfq_15917", "odql_8493", "qg_357", "sfq_5739", "qg_1580", "odql_5501", "dpql_151", "qg_1800", "bb_7364", "qw_16265", "sfq_6175", "jp_1572", "qw_15027", "qz_6596", "odql_7507", "sfq_17997", "qb_3940", "qw_14565", "odql_7762", "qw_3207", "sfq_2355", "sfq_14806", "sfq_10687", "odql_8306", "sfq_16554", "odql_15094", "bb_6092", "qg_2424", "tb_299", "qb_9584", "sfq_7904", "sfq_10502", "tb_1911", "qg_3056", "bt_1811", "bt_371", "qz_5459", "qb_4163", "qb_8707", "odql_14527", "qg_2316", "dpql_1746", "qb_6635", "odql_2749", "odql_4794", "tc_719", "bt_1604", "qf_2264", "sfq_17840", "qb_2148", "dpql_5194", "tc_1944", "qw_15528", "odql_2965", "sfq_14668", "qw_10673", "dpql_4666", "jp_1502", "sfq_6752"]}, {"condition": "dp_sigma_0p5", "score": "mean_answer_nll", "threshold": 0.3716766834259033, "abstain_all": false, "risk_target": 0.1, "selection_scope": "same development data; exploratory", "eligible_ids": ["odql_7980", "qg_749", "odql_11176", "sfq_20838", "sfq_2187", "bb_8725", "qw_10107", "odql_11899", "sfq_8726", "odql_10108", "odql_3325", "tb_1852", "qg_152", "sfq_4423", "sfq_10592", "sfq_18424", "odql_2963", "sfq_12277", "sfq_17015", "bb_9117", "qz_1730", "sfq_23986", "qf_2590", "jp_2935", "sfq_7702", "sfq_23359", "odql_13834", "qg_774", "odql_7981", "wh_2615", "qw_10089", "qw_4795", "dpql_4823", "qb_1953", "qb_292", "odql_9495", "odql_11469", "bb_3733", "odql_14999", "sfq_1917", "qg_527", "qf_3609", "sfq_22986", "sfq_21768", "qb_2880", "bb_2139", "qw_45", "qw_3576", "sfq_6988", "qb_2169", "qf_387", "sfq_3731", "tb_83", "qg_633", "wh_267", "odql_14829", "sfq_15164", "qw_4450", "qb_4066", "qb_3535", "bt_1378", "qg_504", "bt_61", "bb_7375", "bb_1588", "qb_3775", "sfq_15066", "dpql_782", "odql_1772", "qz_6456", "odql_8729", "qf_2569", "odql_8774", "bt_4062", "wh_4082", "qw_2489", "sfq_24884", "qb_8477", "dpql_1994", "odql_13686", "bb_5892", "qg_2735", "odql_2476", "bt_3047", "odql_8663", "qb_4792", "dpql_3600", "sfq_23513", "bt_4028", "sfq_4588", "dpql_2210", "bb_6465", "qb_38", "qz_4435", "odql_4457", "odql_9352", "sfq_19745", "bb_2691", "odql_6624", "qb_8289", "qw_12650", "bt_1819", "sfq_11945", "qz_2822", "qb_7094", "jp_4067", "sfq_6665", "bt_434", "odql_136", "odql_6494", "qz_1047", "qz_4231", "qw_6139", "bt_3299", "qf_277", "odql_249", "dpql_1447", "sfq_13640", "dpql_580", "qf_2048", "dpql_3467", "qb_3556", "qw_2843", "sfq_19473", "jp_4032", "qw_7594", "sfq_3202", "jp_3453", "qw_11112", "qf_345", "dpql_3635", "jp_2435", "odql_13482", "qw_1298", "sfq_15917", "odql_8493", "qg_357", "sfq_5739", "qg_1580", "odql_5501", "dpql_151", "qg_1800", "bb_7364", "qw_16265", "sfq_6175", "jp_1572", "qw_15027", "qz_6596", "odql_7507", "sfq_17997", "qb_3940", "qw_14565", "odql_7762", "qw_3207", "sfq_2355", "sfq_14806", "sfq_10687", "odql_8306", "sfq_16554", "odql_15094", "bb_6092", "qg_2424", "tb_299", "qb_9584", "sfq_7904", "sfq_10502", "tb_1911", "qg_3056", "bt_1811", "bt_371", "qz_5459", "qb_4163", "qb_8707", "odql_14527", "qg_2316", "dpql_1746", "qb_6635", "odql_2749", "odql_4794", "tc_719", "bt_1604", "qf_2264", "sfq_17840", "qb_2148", "dpql_5194", "tc_1944", "qw_15528", "odql_2965", "sfq_14668", "qw_10673", "dpql_4666", "jp_1502", "sfq_6752"]}, {"condition": "dp_sigma_0p5", "score": "discrete_semantic_entropy", "threshold": 0.6108643020548935, "abstain_all": false, "risk_target": 0.1, "selection_scope": "same development data; exploratory", "eligible_ids": ["odql_7980", "qg_749", "odql_11176", "sfq_20838", "sfq_2187", "bb_8725", "qw_10107", "odql_11899", "sfq_8726", "odql_10108", "odql_3325", "tb_1852", "qg_152", "sfq_4423", "sfq_10592", "sfq_18424", "odql_2963", "sfq_12277", "sfq_17015", "bb_9117", "qz_1730", "sfq_23986", "qf_2590", "jp_2935", "sfq_7702", "sfq_23359", "odql_13834", "qg_774", "odql_7981", "wh_2615", "qw_10089", "qw_4795", "dpql_4823", "qb_1953", "qb_292", "odql_9495", "odql_11469", "bb_3733", "odql_14999", "sfq_1917", "qg_527", "qf_3609", "sfq_22986", "sfq_21768", "qb_2880", "bb_2139", "qw_45", "qw_3576", "sfq_6988", "qb_2169", "qf_387", "sfq_3731", "tb_83", "qg_633", "wh_267", "odql_14829", "sfq_15164", "qw_4450", "qb_4066", "qb_3535", "bt_1378", "qg_504", "bt_61", "bb_7375", "bb_1588", "qb_3775", "sfq_15066", "dpql_782", "odql_1772", "qz_6456", "odql_8729", "qf_2569", "odql_8774", "bt_4062", "wh_4082", "qw_2489", "sfq_24884", "qb_8477", "dpql_1994", "odql_13686", "bb_5892", "qg_2735", "odql_2476", "bt_3047", "odql_8663", "qb_4792", "dpql_3600", "sfq_23513", "bt_4028", "sfq_4588", "dpql_2210", "bb_6465", "qb_38", "qz_4435", "odql_4457", "odql_9352", "sfq_19745", "bb_2691", "odql_6624", "qb_8289", "qw_12650", "bt_1819", "sfq_11945", "qz_2822", "qb_7094", "jp_4067", "sfq_6665", "bt_434", "odql_136", "odql_6494", "qz_1047", "qz_4231", "qw_6139", "bt_3299", "qf_277", "odql_249", "dpql_1447", "sfq_13640", "dpql_580", "qf_2048", "dpql_3467", "qb_3556", "qw_2843", "sfq_19473", "jp_4032", "qw_7594", "sfq_3202", "jp_3453", "qw_11112", "qf_345", "dpql_3635", "jp_2435", "odql_13482", "qw_1298", "sfq_15917", "odql_8493", "qg_357", "sfq_5739", "qg_1580", "odql_5501", "dpql_151", "qg_1800", "bb_7364", "qw_16265", "sfq_6175", "jp_1572", "qw_15027", "qz_6596", "odql_7507", "sfq_17997", "qb_3940", "qw_14565", "odql_7762", "qw_3207", "sfq_2355", "sfq_14806", "sfq_10687", "odql_8306", "sfq_16554", "odql_15094", "bb_6092", "qg_2424", "tb_299", "qb_9584", "sfq_7904", "sfq_10502", "tb_1911", "qg_3056", "bt_1811", "bt_371", "qz_5459", "qb_4163", "qb_8707", "odql_14527", "qg_2316", "dpql_1746", "qb_6635", "odql_2749", "odql_4794", "tc_719", "bt_1604", "qf_2264", "sfq_17840", "qb_2148", "dpql_5194", "tc_1944", "qw_15528", "odql_2965", "sfq_14668", "qw_10673", "dpql_4666", "jp_1502", "sfq_6752"]}], "adapter_file_hashes": {"standard_poisson": "720c0d4134db756d082d8d96af09724cdb01a6579a6ddf04352df8a5c713af72", "clipping_poisson": "a06f39ae8a9e8456bf9fe62dbb55e78ec262268e0f524acc12d1c0b46a5293bb", "dp_sigma_0p1": "c3ece7d5a29ef694e44f432dd352df1e00eaa3da7189753a8aca3490d9d184f0", "dp_sigma_0p25": "02a47add1e3172896ad8f2178d66fb838dda6b2621a4fd23d770f3d44ea0bc9e", "dp_sigma_0p5": "0e1341c264c6aed00c183af002ecbdc493b8838739045fe8376aa7f0b0b3b689"}, "development_report": "3ac7a045785c", "threshold_file_sha256": "f4c7655f029ca1011bf28be1af7e1f8469f11ba1d4a56aafe99fc207423bceb8"}, "789": {"suite_config_hash": "44d713a39d51cd99fb088ee4f00f96d6147cea4f9cf5cc626b2b81028f7713a2", "thresholds": [{"condition": "standard_poisson", "score": "mean_token_entropy", "threshold": 1.3031322956085205, "abstain_all": false, "risk_target": 0.1, "selection_scope": "same development data; exploratory", "eligible_ids": ["odql_7980", "qg_749", "odql_11176", "sfq_20838", "sfq_2187", "bb_8725", "qw_10107", "odql_11899", "sfq_8726", "odql_10108", "odql_3325", "tb_1852", "qg_152", "sfq_4423", "sfq_10592", "sfq_18424", "odql_2963", "sfq_12277", "sfq_17015", "bb_9117", "qz_1730", "sfq_23986", "qf_2590", "jp_2935", "sfq_7702", "sfq_23359", "odql_13834", "qg_774", "odql_7981", "wh_2615", "qw_10089", "qw_4795", "dpql_4823", "qb_1953", "qb_292", "odql_9495", "odql_11469", "bb_3733", "odql_14999", "sfq_1917", "qg_527", "qf_3609", "sfq_22986", "sfq_21768", "qb_2880", "bb_2139", "qw_45", "qw_3576", "sfq_6988", "qb_2169", "qf_387", "sfq_3731", "tb_83", "qg_633", "wh_267", "odql_14829", "sfq_15164", "qw_4450", "qb_4066", "qb_3535", "bt_1378", "qg_504", "bt_61", "bb_7375", "bb_1588", "qb_3775", "sfq_15066", "dpql_782", "odql_1772", "qz_6456", "odql_8729", "qf_2569", "odql_8774", "bt_4062", "wh_4082", "qw_2489", "sfq_24884", "qb_8477", "dpql_1994", "odql_13686", "bb_5892", "qg_2735", "odql_2476", "bt_3047", "odql_8663", "qb_4792", "dpql_3600", "sfq_23513", "bt_4028", "sfq_4588", "dpql_2210", "bb_6465", "qb_38", "qz_4435", "odql_4457", "odql_9352", "sfq_19745", "bb_2691", "odql_6624", "qb_8289", "qw_12650", "bt_1819", "sfq_11945", "qz_2822", "qb_7094", "jp_4067", "sfq_6665", "bt_434", "odql_136", "odql_6494", "qz_1047", "qz_4231", "qw_6139", "bt_3299", "qf_277", "odql_249", "dpql_1447", "sfq_13640", "dpql_580", "qf_2048", "dpql_3467", "qb_3556", "qw_2843", "sfq_19473", "jp_4032", "qw_7594", "sfq_3202", "jp_3453", "qw_11112", "qf_345", "dpql_3635", "jp_2435", "odql_13482", "qw_1298", "sfq_15917", "odql_8493", "qg_357", "sfq_5739", "qg_1580", "odql_5501", "dpql_151", "qg_1800", "bb_7364", "qw_16265", "sfq_6175", "jp_1572", "qw_15027", "qz_6596", "odql_7507", "sfq_17997", "qb_3940", "qw_14565", "odql_7762", "qw_3207", "sfq_2355", "sfq_14806", "sfq_10687", "odql_8306", "sfq_16554", "odql_15094", "bb_6092", "qg_2424", "tb_299", "qb_9584", "sfq_7904", "sfq_10502", "tb_1911", "qg_3056", "bt_1811", "bt_371", "qz_5459", "qb_4163", "qb_8707", "odql_14527", "qg_2316", "dpql_1746", "qb_6635", "odql_2749", "odql_4794", "tc_719", "bt_1604", "qf_2264", "sfq_17840", "qb_2148", "dpql_5194", "tc_1944", "qw_15528", "odql_2965", "sfq_14668", "qw_10673", "dpql_4666", "jp_1502", "sfq_6752"]}, {"condition": "standard_poisson", "score": "mean_answer_nll", "threshold": 0.5482726097106934, "abstain_all": false, "risk_target": 0.1, "selection_scope": "same development data; exploratory", "eligible_ids": ["odql_7980", "qg_749", "odql_11176", "sfq_20838", "sfq_2187", "bb_8725", "qw_10107", "odql_11899", "sfq_8726", "odql_10108", "odql_3325", "tb_1852", "qg_152", "sfq_4423", "sfq_10592", "sfq_18424", "odql_2963", "sfq_12277", "sfq_17015", "bb_9117", "qz_1730", "sfq_23986", "qf_2590", "jp_2935", "sfq_7702", "sfq_23359", "odql_13834", "qg_774", "odql_7981", "wh_2615", "qw_10089", "qw_4795", "dpql_4823", "qb_1953", "qb_292", "odql_9495", "odql_11469", "bb_3733", "odql_14999", "sfq_1917", "qg_527", "qf_3609", "sfq_22986", "sfq_21768", "qb_2880", "bb_2139", "qw_45", "qw_3576", "sfq_6988", "qb_2169", "qf_387", "sfq_3731", "tb_83", "qg_633", "wh_267", "odql_14829", "sfq_15164", "qw_4450", "qb_4066", "qb_3535", "bt_1378", "qg_504", "bt_61", "bb_7375", "bb_1588", "qb_3775", "sfq_15066", "dpql_782", "odql_1772", "qz_6456", "odql_8729", "qf_2569", "odql_8774", "bt_4062", "wh_4082", "qw_2489", "sfq_24884", "qb_8477", "dpql_1994", "odql_13686", "bb_5892", "qg_2735", "odql_2476", "bt_3047", "odql_8663", "qb_4792", "dpql_3600", "sfq_23513", "bt_4028", "sfq_4588", "dpql_2210", "bb_6465", "qb_38", "qz_4435", "odql_4457", "odql_9352", "sfq_19745", "bb_2691", "odql_6624", "qb_8289", "qw_12650", "bt_1819", "sfq_11945", "qz_2822", "qb_7094", "jp_4067", "sfq_6665", "bt_434", "odql_136", "odql_6494", "qz_1047", "qz_4231", "qw_6139", "bt_3299", "qf_277", "odql_249", "dpql_1447", "sfq_13640", "dpql_580", "qf_2048", "dpql_3467", "qb_3556", "qw_2843", "sfq_19473", "jp_4032", "qw_7594", "sfq_3202", "jp_3453", "qw_11112", "qf_345", "dpql_3635", "jp_2435", "odql_13482", "qw_1298", "sfq_15917", "odql_8493", "qg_357", "sfq_5739", "qg_1580", "odql_5501", "dpql_151", "qg_1800", "bb_7364", "qw_16265", "sfq_6175", "jp_1572", "qw_15027", "qz_6596", "odql_7507", "sfq_17997", "qb_3940", "qw_14565", "odql_7762", "qw_3207", "sfq_2355", "sfq_14806", "sfq_10687", "odql_8306", "sfq_16554", "odql_15094", "bb_6092", "qg_2424", "tb_299", "qb_9584", "sfq_7904", "sfq_10502", "tb_1911", "qg_3056", "bt_1811", "bt_371", "qz_5459", "qb_4163", "qb_8707", "odql_14527", "qg_2316", "dpql_1746", "qb_6635", "odql_2749", "odql_4794", "tc_719", "bt_1604", "qf_2264", "sfq_17840", "qb_2148", "dpql_5194", "tc_1944", "qw_15528", "odql_2965", "sfq_14668", "qw_10673", "dpql_4666", "jp_1502", "sfq_6752"]}, {"condition": "standard_poisson", "score": "discrete_semantic_entropy", "threshold": 0.8018185525433373, "abstain_all": false, "risk_target": 0.1, "selection_scope": "same development data; exploratory", "eligible_ids": ["odql_7980", "qg_749", "odql_11176", "sfq_20838", "sfq_2187", "bb_8725", "qw_10107", "odql_11899", "sfq_8726", "odql_10108", "odql_3325", "tb_1852", "qg_152", "sfq_4423", "sfq_10592", "sfq_18424", "odql_2963", "sfq_12277", "sfq_17015", "bb_9117", "qz_1730", "sfq_23986", "qf_2590", "jp_2935", "sfq_7702", "sfq_23359", "odql_13834", "qg_774", "odql_7981", "wh_2615", "qw_10089", "qw_4795", "dpql_4823", "qb_1953", "qb_292", "odql_9495", "odql_11469", "bb_3733", "odql_14999", "sfq_1917", "qg_527", "qf_3609", "sfq_22986", "sfq_21768", "qb_2880", "bb_2139", "qw_45", "qw_3576", "sfq_6988", "qb_2169", "qf_387", "sfq_3731", "tb_83", "qg_633", "wh_267", "odql_14829", "sfq_15164", "qw_4450", "qb_4066", "qb_3535", "bt_1378", "qg_504", "bt_61", "bb_7375", "bb_1588", "qb_3775", "sfq_15066", "dpql_782", "odql_1772", "qz_6456", "odql_8729", "qf_2569", "odql_8774", "bt_4062", "wh_4082", "qw_2489", "sfq_24884", "qb_8477", "dpql_1994", "odql_13686", "bb_5892", "qg_2735", "odql_2476", "bt_3047", "odql_8663", "qb_4792", "dpql_3600", "sfq_23513", "bt_4028", "sfq_4588", "dpql_2210", "bb_6465", "qb_38", "qz_4435", "odql_4457", "odql_9352", "sfq_19745", "bb_2691", "odql_6624", "qb_8289", "qw_12650", "bt_1819", "sfq_11945", "qz_2822", "qb_7094", "jp_4067", "sfq_6665", "bt_434", "odql_136", "odql_6494", "qz_1047", "qz_4231", "qw_6139", "bt_3299", "qf_277", "odql_249", "dpql_1447", "sfq_13640", "dpql_580", "qf_2048", "dpql_3467", "qb_3556", "qw_2843", "sfq_19473", "jp_4032", "qw_7594", "sfq_3202", "jp_3453", "qw_11112", "qf_345", "dpql_3635", "jp_2435", "odql_13482", "qw_1298", "sfq_15917", "odql_8493", "qg_357", "sfq_5739", "qg_1580", "odql_5501", "dpql_151", "qg_1800", "bb_7364", "qw_16265", "sfq_6175", "jp_1572", "qw_15027", "qz_6596", "odql_7507", "sfq_17997", "qb_3940", "qw_14565", "odql_7762", "qw_3207", "sfq_2355", "sfq_14806", "sfq_10687", "odql_8306", "sfq_16554", "odql_15094", "bb_6092", "qg_2424", "tb_299", "qb_9584", "sfq_7904", "sfq_10502", "tb_1911", "qg_3056", "bt_1811", "bt_371", "qz_5459", "qb_4163", "qb_8707", "odql_14527", "qg_2316", "dpql_1746", "qb_6635", "odql_2749", "odql_4794", "tc_719", "bt_1604", "qf_2264", "sfq_17840", "qb_2148", "dpql_5194", "tc_1944", "qw_15528", "odql_2965", "sfq_14668", "qw_10673", "dpql_4666", "jp_1502", "sfq_6752"]}, {"condition": "clipping_poisson", "score": "mean_token_entropy", "threshold": 0.8776907920837402, "abstain_all": false, "risk_target": 0.1, "selection_scope": "same development data; exploratory", "eligible_ids": ["odql_7980", "qg_749", "odql_11176", "sfq_20838", "sfq_2187", "bb_8725", "qw_10107", "odql_11899", "sfq_8726", "odql_10108", "odql_3325", "tb_1852", "qg_152", "sfq_4423", "sfq_10592", "sfq_18424", "odql_2963", "sfq_12277", "sfq_17015", "bb_9117", "qz_1730", "sfq_23986", "qf_2590", "jp_2935", "sfq_7702", "sfq_23359", "odql_13834", "qg_774", "odql_7981", "wh_2615", "qw_10089", "qw_4795", "dpql_4823", "qb_1953", "qb_292", "odql_9495", "odql_11469", "bb_3733", "odql_14999", "sfq_1917", "qg_527", "qf_3609", "sfq_22986", "sfq_21768", "qb_2880", "bb_2139", "qw_45", "qw_3576", "sfq_6988", "qb_2169", "qf_387", "sfq_3731", "tb_83", "qg_633", "wh_267", "odql_14829", "sfq_15164", "qw_4450", "qb_4066", "qb_3535", "bt_1378", "qg_504", "bt_61", "bb_7375", "bb_1588", "qb_3775", "sfq_15066", "dpql_782", "odql_1772", "qz_6456", "odql_8729", "qf_2569", "odql_8774", "bt_4062", "wh_4082", "qw_2489", "sfq_24884", "qb_8477", "dpql_1994", "odql_13686", "bb_5892", "qg_2735", "odql_2476", "bt_3047", "odql_8663", "qb_4792", "dpql_3600", "sfq_23513", "bt_4028", "sfq_4588", "dpql_2210", "bb_6465", "qb_38", "qz_4435", "odql_4457", "odql_9352", "sfq_19745", "bb_2691", "odql_6624", "qb_8289", "qw_12650", "bt_1819", "sfq_11945", "qz_2822", "qb_7094", "jp_4067", "sfq_6665", "bt_434", "odql_136", "odql_6494", "qz_1047", "qz_4231", "qw_6139", "bt_3299", "qf_277", "odql_249", "dpql_1447", "sfq_13640", "dpql_580", "qf_2048", "dpql_3467", "qb_3556", "qw_2843", "sfq_19473", "jp_4032", "qw_7594", "sfq_3202", "jp_3453", "qw_11112", "qf_345", "dpql_3635", "jp_2435", "odql_13482", "qw_1298", "sfq_15917", "odql_8493", "qg_357", "sfq_5739", "qg_1580", "odql_5501", "dpql_151", "qg_1800", "bb_7364", "qw_16265", "sfq_6175", "jp_1572", "qw_15027", "qz_6596", "odql_7507", "sfq_17997", "qb_3940", "qw_14565", "odql_7762", "qw_3207", "sfq_2355", "sfq_14806", "sfq_10687", "odql_8306", "sfq_16554", "odql_15094", "bb_6092", "qg_2424", "tb_299", "qb_9584", "sfq_7904", "sfq_10502", "tb_1911", "qg_3056", "bt_1811", "bt_371", "qz_5459", "qb_4163", "qb_8707", "odql_14527", "qg_2316", "dpql_1746", "qb_6635", "odql_2749", "odql_4794", "tc_719", "bt_1604", "qf_2264", "sfq_17840", "qb_2148", "dpql_5194", "tc_1944", "qw_15528", "odql_2965", "sfq_14668", "qw_10673", "dpql_4666", "jp_1502", "sfq_6752"]}, {"condition": "clipping_poisson", "score": "mean_answer_nll", "threshold": 0.31445854902267456, "abstain_all": false, "risk_target": 0.1, "selection_scope": "same development data; exploratory", "eligible_ids": ["odql_7980", "qg_749", "odql_11176", "sfq_20838", "sfq_2187", "bb_8725", "qw_10107", "odql_11899", "sfq_8726", "odql_10108", "odql_3325", "tb_1852", "qg_152", "sfq_4423", "sfq_10592", "sfq_18424", "odql_2963", "sfq_12277", "sfq_17015", "bb_9117", "qz_1730", "sfq_23986", "qf_2590", "jp_2935", "sfq_7702", "sfq_23359", "odql_13834", "qg_774", "odql_7981", "wh_2615", "qw_10089", "qw_4795", "dpql_4823", "qb_1953", "qb_292", "odql_9495", "odql_11469", "bb_3733", "odql_14999", "sfq_1917", "qg_527", "qf_3609", "sfq_22986", "sfq_21768", "qb_2880", "bb_2139", "qw_45", "qw_3576", "sfq_6988", "qb_2169", "qf_387", "sfq_3731", "tb_83", "qg_633", "wh_267", "odql_14829", "sfq_15164", "qw_4450", "qb_4066", "qb_3535", "bt_1378", "qg_504", "bt_61", "bb_7375", "bb_1588", "qb_3775", "sfq_15066", "dpql_782", "odql_1772", "qz_6456", "odql_8729", "qf_2569", "odql_8774", "bt_4062", "wh_4082", "qw_2489", "sfq_24884", "qb_8477", "dpql_1994", "odql_13686", "bb_5892", "qg_2735", "odql_2476", "bt_3047", "odql_8663", "qb_4792", "dpql_3600", "sfq_23513", "bt_4028", "sfq_4588", "dpql_2210", "bb_6465", "qb_38", "qz_4435", "odql_4457", "odql_9352", "sfq_19745", "bb_2691", "odql_6624", "qb_8289", "qw_12650", "bt_1819", "sfq_11945", "qz_2822", "qb_7094", "jp_4067", "sfq_6665", "bt_434", "odql_136", "odql_6494", "qz_1047", "qz_4231", "qw_6139", "bt_3299", "qf_277", "odql_249", "dpql_1447", "sfq_13640", "dpql_580", "qf_2048", "dpql_3467", "qb_3556", "qw_2843", "sfq_19473", "jp_4032", "qw_7594", "sfq_3202", "jp_3453", "qw_11112", "qf_345", "dpql_3635", "jp_2435", "odql_13482", "qw_1298", "sfq_15917", "odql_8493", "qg_357", "sfq_5739", "qg_1580", "odql_5501", "dpql_151", "qg_1800", "bb_7364", "qw_16265", "sfq_6175", "jp_1572", "qw_15027", "qz_6596", "odql_7507", "sfq_17997", "qb_3940", "qw_14565", "odql_7762", "qw_3207", "sfq_2355", "sfq_14806", "sfq_10687", "odql_8306", "sfq_16554", "odql_15094", "bb_6092", "qg_2424", "tb_299", "qb_9584", "sfq_7904", "sfq_10502", "tb_1911", "qg_3056", "bt_1811", "bt_371", "qz_5459", "qb_4163", "qb_8707", "odql_14527", "qg_2316", "dpql_1746", "qb_6635", "odql_2749", "odql_4794", "tc_719", "bt_1604", "qf_2264", "sfq_17840", "qb_2148", "dpql_5194", "tc_1944", "qw_15528", "odql_2965", "sfq_14668", "qw_10673", "dpql_4666", "jp_1502", "sfq_6752"]}, {"condition": "clipping_poisson", "score": "discrete_semantic_entropy", "threshold": 0.6108643020548935, "abstain_all": false, "risk_target": 0.1, "selection_scope": "same development data; exploratory", "eligible_ids": ["odql_7980", "qg_749", "odql_11176", "sfq_20838", "sfq_2187", "bb_8725", "qw_10107", "odql_11899", "sfq_8726", "odql_10108", "odql_3325", "tb_1852", "qg_152", "sfq_4423", "sfq_10592", "sfq_18424", "odql_2963", "sfq_12277", "sfq_17015", "bb_9117", "qz_1730", "sfq_23986", "qf_2590", "jp_2935", "sfq_7702", "sfq_23359", "odql_13834", "qg_774", "odql_7981", "wh_2615", "qw_10089", "qw_4795", "dpql_4823", "qb_1953", "qb_292", "odql_9495", "odql_11469", "bb_3733", "odql_14999", "sfq_1917", "qg_527", "qf_3609", "sfq_22986", "sfq_21768", "qb_2880", "bb_2139", "qw_45", "qw_3576", "sfq_6988", "qb_2169", "qf_387", "sfq_3731", "tb_83", "qg_633", "wh_267", "odql_14829", "sfq_15164", "qw_4450", "qb_4066", "qb_3535", "bt_1378", "qg_504", "bt_61", "bb_7375", "bb_1588", "qb_3775", "sfq_15066", "dpql_782", "odql_1772", "qz_6456", "odql_8729", "qf_2569", "odql_8774", "bt_4062", "wh_4082", "qw_2489", "sfq_24884", "qb_8477", "dpql_1994", "odql_13686", "bb_5892", "qg_2735", "odql_2476", "bt_3047", "odql_8663", "qb_4792", "dpql_3600", "sfq_23513", "bt_4028", "sfq_4588", "dpql_2210", "bb_6465", "qb_38", "qz_4435", "odql_4457", "odql_9352", "sfq_19745", "bb_2691", "odql_6624", "qb_8289", "qw_12650", "bt_1819", "sfq_11945", "qz_2822", "qb_7094", "jp_4067", "sfq_6665", "bt_434", "odql_136", "odql_6494", "qz_1047", "qz_4231", "qw_6139", "bt_3299", "qf_277", "odql_249", "dpql_1447", "sfq_13640", "dpql_580", "qf_2048", "dpql_3467", "qb_3556", "qw_2843", "sfq_19473", "jp_4032", "qw_7594", "sfq_3202", "jp_3453", "qw_11112", "qf_345", "dpql_3635", "jp_2435", "odql_13482", "qw_1298", "sfq_15917", "odql_8493", "qg_357", "sfq_5739", "qg_1580", "odql_5501", "dpql_151", "qg_1800", "bb_7364", "qw_16265", "sfq_6175", "jp_1572", "qw_15027", "qz_6596", "odql_7507", "sfq_17997", "qb_3940", "qw_14565", "odql_7762", "qw_3207", "sfq_2355", "sfq_14806", "sfq_10687", "odql_8306", "sfq_16554", "odql_15094", "bb_6092", "qg_2424", "tb_299", "qb_9584", "sfq_7904", "sfq_10502", "tb_1911", "qg_3056", "bt_1811", "bt_371", "qz_5459", "qb_4163", "qb_8707", "odql_14527", "qg_2316", "dpql_1746", "qb_6635", "odql_2749", "odql_4794", "tc_719", "bt_1604", "qf_2264", "sfq_17840", "qb_2148", "dpql_5194", "tc_1944", "qw_15528", "odql_2965", "sfq_14668", "qw_10673", "dpql_4666", "jp_1502", "sfq_6752"]}, {"condition": "dp_sigma_0p1", "score": "mean_token_entropy", "threshold": 1.346727728843689, "abstain_all": false, "risk_target": 0.1, "selection_scope": "same development data; exploratory", "eligible_ids": ["odql_7980", "qg_749", "odql_11176", "sfq_20838", "sfq_2187", "bb_8725", "qw_10107", "odql_11899", "sfq_8726", "odql_10108", "odql_3325", "tb_1852", "qg_152", "sfq_4423", "sfq_10592", "sfq_18424", "odql_2963", "sfq_12277", "sfq_17015", "bb_9117", "qz_1730", "sfq_23986", "qf_2590", "jp_2935", "sfq_7702", "sfq_23359", "odql_13834", "qg_774", "odql_7981", "wh_2615", "qw_10089", "qw_4795", "dpql_4823", "qb_1953", "qb_292", "odql_9495", "odql_11469", "bb_3733", "odql_14999", "sfq_1917", "qg_527", "qf_3609", "sfq_22986", "sfq_21768", "qb_2880", "bb_2139", "qw_45", "qw_3576", "sfq_6988", "qb_2169", "qf_387", "sfq_3731", "tb_83", "qg_633", "wh_267", "odql_14829", "sfq_15164", "qw_4450", "qb_4066", "qb_3535", "bt_1378", "qg_504", "bt_61", "bb_7375", "bb_1588", "qb_3775", "sfq_15066", "dpql_782", "odql_1772", "qz_6456", "odql_8729", "qf_2569", "odql_8774", "bt_4062", "wh_4082", "qw_2489", "sfq_24884", "qb_8477", "dpql_1994", "odql_13686", "bb_5892", "qg_2735", "odql_2476", "bt_3047", "odql_8663", "qb_4792", "dpql_3600", "sfq_23513", "bt_4028", "sfq_4588", "dpql_2210", "bb_6465", "qb_38", "qz_4435", "odql_4457", "odql_9352", "sfq_19745", "bb_2691", "odql_6624", "qb_8289", "qw_12650", "bt_1819", "sfq_11945", "qz_2822", "qb_7094", "jp_4067", "sfq_6665", "bt_434", "odql_136", "odql_6494", "qz_1047", "qz_4231", "qw_6139", "bt_3299", "qf_277", "odql_249", "dpql_1447", "sfq_13640", "dpql_580", "qf_2048", "dpql_3467", "qb_3556", "qw_2843", "sfq_19473", "jp_4032", "qw_7594", "sfq_3202", "jp_3453", "qw_11112", "qf_345", "dpql_3635", "jp_2435", "odql_13482", "qw_1298", "sfq_15917", "odql_8493", "qg_357", "sfq_5739", "qg_1580", "odql_5501", "dpql_151", "qg_1800", "bb_7364", "qw_16265", "sfq_6175", "jp_1572", "qw_15027", "qz_6596", "odql_7507", "sfq_17997", "qb_3940", "qw_14565", "odql_7762", "qw_3207", "sfq_2355", "sfq_14806", "sfq_10687", "odql_8306", "sfq_16554", "odql_15094", "bb_6092", "qg_2424", "tb_299", "qb_9584", "sfq_7904", "sfq_10502", "tb_1911", "qg_3056", "bt_1811", "bt_371", "qz_5459", "qb_4163", "qb_8707", "odql_14527", "qg_2316", "dpql_1746", "qb_6635", "odql_2749", "odql_4794", "tc_719", "bt_1604", "qf_2264", "sfq_17840", "qb_2148", "dpql_5194", "tc_1944", "qw_15528", "odql_2965", "sfq_14668", "qw_10673", "dpql_4666", "jp_1502", "sfq_6752"]}, {"condition": "dp_sigma_0p1", "score": "mean_answer_nll", "threshold": 0.5371167063713074, "abstain_all": false, "risk_target": 0.1, "selection_scope": "same development data; exploratory", "eligible_ids": ["odql_7980", "qg_749", "odql_11176", "sfq_20838", "sfq_2187", "bb_8725", "qw_10107", "odql_11899", "sfq_8726", "odql_10108", "odql_3325", "tb_1852", "qg_152", "sfq_4423", "sfq_10592", "sfq_18424", "odql_2963", "sfq_12277", "sfq_17015", "bb_9117", "qz_1730", "sfq_23986", "qf_2590", "jp_2935", "sfq_7702", "sfq_23359", "odql_13834", "qg_774", "odql_7981", "wh_2615", "qw_10089", "qw_4795", "dpql_4823", "qb_1953", "qb_292", "odql_9495", "odql_11469", "bb_3733", "odql_14999", "sfq_1917", "qg_527", "qf_3609", "sfq_22986", "sfq_21768", "qb_2880", "bb_2139", "qw_45", "qw_3576", "sfq_6988", "qb_2169", "qf_387", "sfq_3731", "tb_83", "qg_633", "wh_267", "odql_14829", "sfq_15164", "qw_4450", "qb_4066", "qb_3535", "bt_1378", "qg_504", "bt_61", "bb_7375", "bb_1588", "qb_3775", "sfq_15066", "dpql_782", "odql_1772", "qz_6456", "odql_8729", "qf_2569", "odql_8774", "bt_4062", "wh_4082", "qw_2489", "sfq_24884", "qb_8477", "dpql_1994", "odql_13686", "bb_5892", "qg_2735", "odql_2476", "bt_3047", "odql_8663", "qb_4792", "dpql_3600", "sfq_23513", "bt_4028", "sfq_4588", "dpql_2210", "bb_6465", "qb_38", "qz_4435", "odql_4457", "odql_9352", "sfq_19745", "bb_2691", "odql_6624", "qb_8289", "qw_12650", "bt_1819", "sfq_11945", "qz_2822", "qb_7094", "jp_4067", "sfq_6665", "bt_434", "odql_136", "odql_6494", "qz_1047", "qz_4231", "qw_6139", "bt_3299", "qf_277", "odql_249", "dpql_1447", "sfq_13640", "dpql_580", "qf_2048", "dpql_3467", "qb_3556", "qw_2843", "sfq_19473", "jp_4032", "qw_7594", "sfq_3202", "jp_3453", "qw_11112", "qf_345", "dpql_3635", "jp_2435", "odql_13482", "qw_1298", "sfq_15917", "odql_8493", "qg_357", "sfq_5739", "qg_1580", "odql_5501", "dpql_151", "qg_1800", "bb_7364", "qw_16265", "sfq_6175", "jp_1572", "qw_15027", "qz_6596", "odql_7507", "sfq_17997", "qb_3940", "qw_14565", "odql_7762", "qw_3207", "sfq_2355", "sfq_14806", "sfq_10687", "odql_8306", "sfq_16554", "odql_15094", "bb_6092", "qg_2424", "tb_299", "qb_9584", "sfq_7904", "sfq_10502", "tb_1911", "qg_3056", "bt_1811", "bt_371", "qz_5459", "qb_4163", "qb_8707", "odql_14527", "qg_2316", "dpql_1746", "qb_6635", "odql_2749", "odql_4794", "tc_719", "bt_1604", "qf_2264", "sfq_17840", "qb_2148", "dpql_5194", "tc_1944", "qw_15528", "odql_2965", "sfq_14668", "qw_10673", "dpql_4666", "jp_1502", "sfq_6752"]}, {"condition": "dp_sigma_0p1", "score": "discrete_semantic_entropy", "threshold": 0.8018185525433373, "abstain_all": false, "risk_target": 0.1, "selection_scope": "same development data; exploratory", "eligible_ids": ["odql_7980", "qg_749", "odql_11176", "sfq_20838", "sfq_2187", "bb_8725", "qw_10107", "odql_11899", "sfq_8726", "odql_10108", "odql_3325", "tb_1852", "qg_152", "sfq_4423", "sfq_10592", "sfq_18424", "odql_2963", "sfq_12277", "sfq_17015", "bb_9117", "qz_1730", "sfq_23986", "qf_2590", "jp_2935", "sfq_7702", "sfq_23359", "odql_13834", "qg_774", "odql_7981", "wh_2615", "qw_10089", "qw_4795", "dpql_4823", "qb_1953", "qb_292", "odql_9495", "odql_11469", "bb_3733", "odql_14999", "sfq_1917", "qg_527", "qf_3609", "sfq_22986", "sfq_21768", "qb_2880", "bb_2139", "qw_45", "qw_3576", "sfq_6988", "qb_2169", "qf_387", "sfq_3731", "tb_83", "qg_633", "wh_267", "odql_14829", "sfq_15164", "qw_4450", "qb_4066", "qb_3535", "bt_1378", "qg_504", "bt_61", "bb_7375", "bb_1588", "qb_3775", "sfq_15066", "dpql_782", "odql_1772", "qz_6456", "odql_8729", "qf_2569", "odql_8774", "bt_4062", "wh_4082", "qw_2489", "sfq_24884", "qb_8477", "dpql_1994", "odql_13686", "bb_5892", "qg_2735", "odql_2476", "bt_3047", "odql_8663", "qb_4792", "dpql_3600", "sfq_23513", "bt_4028", "sfq_4588", "dpql_2210", "bb_6465", "qb_38", "qz_4435", "odql_4457", "odql_9352", "sfq_19745", "bb_2691", "odql_6624", "qb_8289", "qw_12650", "bt_1819", "sfq_11945", "qz_2822", "qb_7094", "jp_4067", "sfq_6665", "bt_434", "odql_136", "odql_6494", "qz_1047", "qz_4231", "qw_6139", "bt_3299", "qf_277", "odql_249", "dpql_1447", "sfq_13640", "dpql_580", "qf_2048", "dpql_3467", "qb_3556", "qw_2843", "sfq_19473", "jp_4032", "qw_7594", "sfq_3202", "jp_3453", "qw_11112", "qf_345", "dpql_3635", "jp_2435", "odql_13482", "qw_1298", "sfq_15917", "odql_8493", "qg_357", "sfq_5739", "qg_1580", "odql_5501", "dpql_151", "qg_1800", "bb_7364", "qw_16265", "sfq_6175", "jp_1572", "qw_15027", "qz_6596", "odql_7507", "sfq_17997", "qb_3940", "qw_14565", "odql_7762", "qw_3207", "sfq_2355", "sfq_14806", "sfq_10687", "odql_8306", "sfq_16554", "odql_15094", "bb_6092", "qg_2424", "tb_299", "qb_9584", "sfq_7904", "sfq_10502", "tb_1911", "qg_3056", "bt_1811", "bt_371", "qz_5459", "qb_4163", "qb_8707", "odql_14527", "qg_2316", "dpql_1746", "qb_6635", "odql_2749", "odql_4794", "tc_719", "bt_1604", "qf_2264", "sfq_17840", "qb_2148", "dpql_5194", "tc_1944", "qw_15528", "odql_2965", "sfq_14668", "qw_10673", "dpql_4666", "jp_1502", "sfq_6752"]}, {"condition": "dp_sigma_0p25", "score": "mean_token_entropy", "threshold": 1.000688910484314, "abstain_all": false, "risk_target": 0.1, "selection_scope": "same development data; exploratory", "eligible_ids": ["odql_7980", "qg_749", "odql_11176", "sfq_20838", "sfq_2187", "bb_8725", "qw_10107", "odql_11899", "sfq_8726", "odql_10108", "odql_3325", "tb_1852", "qg_152", "sfq_4423", "sfq_10592", "sfq_18424", "odql_2963", "sfq_12277", "sfq_17015", "bb_9117", "qz_1730", "sfq_23986", "qf_2590", "jp_2935", "sfq_7702", "sfq_23359", "odql_13834", "qg_774", "odql_7981", "wh_2615", "qw_10089", "qw_4795", "dpql_4823", "qb_1953", "qb_292", "odql_9495", "odql_11469", "bb_3733", "odql_14999", "sfq_1917", "qg_527", "qf_3609", "sfq_22986", "sfq_21768", "qb_2880", "bb_2139", "qw_45", "qw_3576", "sfq_6988", "qb_2169", "qf_387", "sfq_3731", "tb_83", "qg_633", "wh_267", "odql_14829", "sfq_15164", "qw_4450", "qb_4066", "qb_3535", "bt_1378", "qg_504", "bt_61", "bb_7375", "bb_1588", "qb_3775", "sfq_15066", "dpql_782", "odql_1772", "qz_6456", "odql_8729", "qf_2569", "odql_8774", "bt_4062", "wh_4082", "qw_2489", "sfq_24884", "qb_8477", "dpql_1994", "odql_13686", "bb_5892", "qg_2735", "odql_2476", "bt_3047", "odql_8663", "qb_4792", "dpql_3600", "sfq_23513", "bt_4028", "sfq_4588", "dpql_2210", "bb_6465", "qb_38", "qz_4435", "odql_4457", "odql_9352", "sfq_19745", "bb_2691", "odql_6624", "qb_8289", "qw_12650", "bt_1819", "sfq_11945", "qz_2822", "qb_7094", "jp_4067", "sfq_6665", "bt_434", "odql_136", "odql_6494", "qz_1047", "qz_4231", "qw_6139", "bt_3299", "qf_277", "odql_249", "dpql_1447", "sfq_13640", "dpql_580", "qf_2048", "dpql_3467", "qb_3556", "qw_2843", "sfq_19473", "jp_4032", "qw_7594", "sfq_3202", "jp_3453", "qw_11112", "qf_345", "dpql_3635", "jp_2435", "odql_13482", "qw_1298", "sfq_15917", "odql_8493", "qg_357", "sfq_5739", "qg_1580", "odql_5501", "dpql_151", "qg_1800", "bb_7364", "qw_16265", "sfq_6175", "jp_1572", "qw_15027", "qz_6596", "odql_7507", "sfq_17997", "qb_3940", "qw_14565", "odql_7762", "qw_3207", "sfq_2355", "sfq_14806", "sfq_10687", "odql_8306", "sfq_16554", "odql_15094", "bb_6092", "qg_2424", "tb_299", "qb_9584", "sfq_7904", "sfq_10502", "tb_1911", "qg_3056", "bt_1811", "bt_371", "qz_5459", "qb_4163", "qb_8707", "odql_14527", "qg_2316", "dpql_1746", "qb_6635", "odql_2749", "odql_4794", "tc_719", "bt_1604", "qf_2264", "sfq_17840", "qb_2148", "dpql_5194", "tc_1944", "qw_15528", "odql_2965", "sfq_14668", "qw_10673", "dpql_4666", "jp_1502", "sfq_6752"]}, {"condition": "dp_sigma_0p25", "score": "mean_answer_nll", "threshold": 0.4347389340400696, "abstain_all": false, "risk_target": 0.1, "selection_scope": "same development data; exploratory", "eligible_ids": ["odql_7980", "qg_749", "odql_11176", "sfq_20838", "sfq_2187", "bb_8725", "qw_10107", "odql_11899", "sfq_8726", "odql_10108", "odql_3325", "tb_1852", "qg_152", "sfq_4423", "sfq_10592", "sfq_18424", "odql_2963", "sfq_12277", "sfq_17015", "bb_9117", "qz_1730", "sfq_23986", "qf_2590", "jp_2935", "sfq_7702", "sfq_23359", "odql_13834", "qg_774", "odql_7981", "wh_2615", "qw_10089", "qw_4795", "dpql_4823", "qb_1953", "qb_292", "odql_9495", "odql_11469", "bb_3733", "odql_14999", "sfq_1917", "qg_527", "qf_3609", "sfq_22986", "sfq_21768", "qb_2880", "bb_2139", "qw_45", "qw_3576", "sfq_6988", "qb_2169", "qf_387", "sfq_3731", "tb_83", "qg_633", "wh_267", "odql_14829", "sfq_15164", "qw_4450", "qb_4066", "qb_3535", "bt_1378", "qg_504", "bt_61", "bb_7375", "bb_1588", "qb_3775", "sfq_15066", "dpql_782", "odql_1772", "qz_6456", "odql_8729", "qf_2569", "odql_8774", "bt_4062", "wh_4082", "qw_2489", "sfq_24884", "qb_8477", "dpql_1994", "odql_13686", "bb_5892", "qg_2735", "odql_2476", "bt_3047", "odql_8663", "qb_4792", "dpql_3600", "sfq_23513", "bt_4028", "sfq_4588", "dpql_2210", "bb_6465", "qb_38", "qz_4435", "odql_4457", "odql_9352", "sfq_19745", "bb_2691", "odql_6624", "qb_8289", "qw_12650", "bt_1819", "sfq_11945", "qz_2822", "qb_7094", "jp_4067", "sfq_6665", "bt_434", "odql_136", "odql_6494", "qz_1047", "qz_4231", "qw_6139", "bt_3299", "qf_277", "odql_249", "dpql_1447", "sfq_13640", "dpql_580", "qf_2048", "dpql_3467", "qb_3556", "qw_2843", "sfq_19473", "jp_4032", "qw_7594", "sfq_3202", "jp_3453", "qw_11112", "qf_345", "dpql_3635", "jp_2435", "odql_13482", "qw_1298", "sfq_15917", "odql_8493", "qg_357", "sfq_5739", "qg_1580", "odql_5501", "dpql_151", "qg_1800", "bb_7364", "qw_16265", "sfq_6175", "jp_1572", "qw_15027", "qz_6596", "odql_7507", "sfq_17997", "qb_3940", "qw_14565", "odql_7762", "qw_3207", "sfq_2355", "sfq_14806", "sfq_10687", "odql_8306", "sfq_16554", "odql_15094", "bb_6092", "qg_2424", "tb_299", "qb_9584", "sfq_7904", "sfq_10502", "tb_1911", "qg_3056", "bt_1811", "bt_371", "qz_5459", "qb_4163", "qb_8707", "odql_14527", "qg_2316", "dpql_1746", "qb_6635", "odql_2749", "odql_4794", "tc_719", "bt_1604", "qf_2264", "sfq_17840", "qb_2148", "dpql_5194", "tc_1944", "qw_15528", "odql_2965", "sfq_14668", "qw_10673", "dpql_4666", "jp_1502", "sfq_6752"]}, {"condition": "dp_sigma_0p25", "score": "discrete_semantic_entropy", "threshold": 0.6108643020548935, "abstain_all": false, "risk_target": 0.1, "selection_scope": "same development data; exploratory", "eligible_ids": ["odql_7980", "qg_749", "odql_11176", "sfq_20838", "sfq_2187", "bb_8725", "qw_10107", "odql_11899", "sfq_8726", "odql_10108", "odql_3325", "tb_1852", "qg_152", "sfq_4423", "sfq_10592", "sfq_18424", "odql_2963", "sfq_12277", "sfq_17015", "bb_9117", "qz_1730", "sfq_23986", "qf_2590", "jp_2935", "sfq_7702", "sfq_23359", "odql_13834", "qg_774", "odql_7981", "wh_2615", "qw_10089", "qw_4795", "dpql_4823", "qb_1953", "qb_292", "odql_9495", "odql_11469", "bb_3733", "odql_14999", "sfq_1917", "qg_527", "qf_3609", "sfq_22986", "sfq_21768", "qb_2880", "bb_2139", "qw_45", "qw_3576", "sfq_6988", "qb_2169", "qf_387", "sfq_3731", "tb_83", "qg_633", "wh_267", "odql_14829", "sfq_15164", "qw_4450", "qb_4066", "qb_3535", "bt_1378", "qg_504", "bt_61", "bb_7375", "bb_1588", "qb_3775", "sfq_15066", "dpql_782", "odql_1772", "qz_6456", "odql_8729", "qf_2569", "odql_8774", "bt_4062", "wh_4082", "qw_2489", "sfq_24884", "qb_8477", "dpql_1994", "odql_13686", "bb_5892", "qg_2735", "odql_2476", "bt_3047", "odql_8663", "qb_4792", "dpql_3600", "sfq_23513", "bt_4028", "sfq_4588", "dpql_2210", "bb_6465", "qb_38", "qz_4435", "odql_4457", "odql_9352", "sfq_19745", "bb_2691", "odql_6624", "qb_8289", "qw_12650", "bt_1819", "sfq_11945", "qz_2822", "qb_7094", "jp_4067", "sfq_6665", "bt_434", "odql_136", "odql_6494", "qz_1047", "qz_4231", "qw_6139", "bt_3299", "qf_277", "odql_249", "dpql_1447", "sfq_13640", "dpql_580", "qf_2048", "dpql_3467", "qb_3556", "qw_2843", "sfq_19473", "jp_4032", "qw_7594", "sfq_3202", "jp_3453", "qw_11112", "qf_345", "dpql_3635", "jp_2435", "odql_13482", "qw_1298", "sfq_15917", "odql_8493", "qg_357", "sfq_5739", "qg_1580", "odql_5501", "dpql_151", "qg_1800", "bb_7364", "qw_16265", "sfq_6175", "jp_1572", "qw_15027", "qz_6596", "odql_7507", "sfq_17997", "qb_3940", "qw_14565", "odql_7762", "qw_3207", "sfq_2355", "sfq_14806", "sfq_10687", "odql_8306", "sfq_16554", "odql_15094", "bb_6092", "qg_2424", "tb_299", "qb_9584", "sfq_7904", "sfq_10502", "tb_1911", "qg_3056", "bt_1811", "bt_371", "qz_5459", "qb_4163", "qb_8707", "odql_14527", "qg_2316", "dpql_1746", "qb_6635", "odql_2749", "odql_4794", "tc_719", "bt_1604", "qf_2264", "sfq_17840", "qb_2148", "dpql_5194", "tc_1944", "qw_15528", "odql_2965", "sfq_14668", "qw_10673", "dpql_4666", "jp_1502", "sfq_6752"]}, {"condition": "dp_sigma_0p5", "score": "mean_token_entropy", "threshold": 0.9799665212631226, "abstain_all": false, "risk_target": 0.1, "selection_scope": "same development data; exploratory", "eligible_ids": ["odql_7980", "qg_749", "odql_11176", "sfq_20838", "sfq_2187", "bb_8725", "qw_10107", "odql_11899", "sfq_8726", "odql_10108", "odql_3325", "tb_1852", "qg_152", "sfq_4423", "sfq_10592", "sfq_18424", "odql_2963", "sfq_12277", "sfq_17015", "bb_9117", "qz_1730", "sfq_23986", "qf_2590", "jp_2935", "sfq_7702", "sfq_23359", "odql_13834", "qg_774", "odql_7981", "wh_2615", "qw_10089", "qw_4795", "dpql_4823", "qb_1953", "qb_292", "odql_9495", "odql_11469", "bb_3733", "odql_14999", "sfq_1917", "qg_527", "qf_3609", "sfq_22986", "sfq_21768", "qb_2880", "bb_2139", "qw_45", "qw_3576", "sfq_6988", "qb_2169", "qf_387", "sfq_3731", "tb_83", "qg_633", "wh_267", "odql_14829", "sfq_15164", "qw_4450", "qb_4066", "qb_3535", "bt_1378", "qg_504", "bt_61", "bb_7375", "bb_1588", "qb_3775", "sfq_15066", "dpql_782", "odql_1772", "qz_6456", "odql_8729", "qf_2569", "odql_8774", "bt_4062", "wh_4082", "qw_2489", "sfq_24884", "qb_8477", "dpql_1994", "odql_13686", "bb_5892", "qg_2735", "odql_2476", "bt_3047", "odql_8663", "qb_4792", "dpql_3600", "sfq_23513", "bt_4028", "sfq_4588", "dpql_2210", "bb_6465", "qb_38", "qz_4435", "odql_4457", "odql_9352", "sfq_19745", "bb_2691", "odql_6624", "qb_8289", "qw_12650", "bt_1819", "sfq_11945", "qz_2822", "qb_7094", "jp_4067", "sfq_6665", "bt_434", "odql_136", "odql_6494", "qz_1047", "qz_4231", "qw_6139", "bt_3299", "qf_277", "odql_249", "dpql_1447", "sfq_13640", "dpql_580", "qf_2048", "dpql_3467", "qb_3556", "qw_2843", "sfq_19473", "jp_4032", "qw_7594", "sfq_3202", "jp_3453", "qw_11112", "qf_345", "dpql_3635", "jp_2435", "odql_13482", "qw_1298", "sfq_15917", "odql_8493", "qg_357", "sfq_5739", "qg_1580", "odql_5501", "dpql_151", "qg_1800", "bb_7364", "qw_16265", "sfq_6175", "jp_1572", "qw_15027", "qz_6596", "odql_7507", "sfq_17997", "qb_3940", "qw_14565", "odql_7762", "qw_3207", "sfq_2355", "sfq_14806", "sfq_10687", "odql_8306", "sfq_16554", "odql_15094", "bb_6092", "qg_2424", "tb_299", "qb_9584", "sfq_7904", "sfq_10502", "tb_1911", "qg_3056", "bt_1811", "bt_371", "qz_5459", "qb_4163", "qb_8707", "odql_14527", "qg_2316", "dpql_1746", "qb_6635", "odql_2749", "odql_4794", "tc_719", "bt_1604", "qf_2264", "sfq_17840", "qb_2148", "dpql_5194", "tc_1944", "qw_15528", "odql_2965", "sfq_14668", "qw_10673", "dpql_4666", "jp_1502", "sfq_6752"]}, {"condition": "dp_sigma_0p5", "score": "mean_answer_nll", "threshold": 0.3957253098487854, "abstain_all": false, "risk_target": 0.1, "selection_scope": "same development data; exploratory", "eligible_ids": ["odql_7980", "qg_749", "odql_11176", "sfq_20838", "sfq_2187", "bb_8725", "qw_10107", "odql_11899", "sfq_8726", "odql_10108", "odql_3325", "tb_1852", "qg_152", "sfq_4423", "sfq_10592", "sfq_18424", "odql_2963", "sfq_12277", "sfq_17015", "bb_9117", "qz_1730", "sfq_23986", "qf_2590", "jp_2935", "sfq_7702", "sfq_23359", "odql_13834", "qg_774", "odql_7981", "wh_2615", "qw_10089", "qw_4795", "dpql_4823", "qb_1953", "qb_292", "odql_9495", "odql_11469", "bb_3733", "odql_14999", "sfq_1917", "qg_527", "qf_3609", "sfq_22986", "sfq_21768", "qb_2880", "bb_2139", "qw_45", "qw_3576", "sfq_6988", "qb_2169", "qf_387", "sfq_3731", "tb_83", "qg_633", "wh_267", "odql_14829", "sfq_15164", "qw_4450", "qb_4066", "qb_3535", "bt_1378", "qg_504", "bt_61", "bb_7375", "bb_1588", "qb_3775", "sfq_15066", "dpql_782", "odql_1772", "qz_6456", "odql_8729", "qf_2569", "odql_8774", "bt_4062", "wh_4082", "qw_2489", "sfq_24884", "qb_8477", "dpql_1994", "odql_13686", "bb_5892", "qg_2735", "odql_2476", "bt_3047", "odql_8663", "qb_4792", "dpql_3600", "sfq_23513", "bt_4028", "sfq_4588", "dpql_2210", "bb_6465", "qb_38", "qz_4435", "odql_4457", "odql_9352", "sfq_19745", "bb_2691", "odql_6624", "qb_8289", "qw_12650", "bt_1819", "sfq_11945", "qz_2822", "qb_7094", "jp_4067", "sfq_6665", "bt_434", "odql_136", "odql_6494", "qz_1047", "qz_4231", "qw_6139", "bt_3299", "qf_277", "odql_249", "dpql_1447", "sfq_13640", "dpql_580", "qf_2048", "dpql_3467", "qb_3556", "qw_2843", "sfq_19473", "jp_4032", "qw_7594", "sfq_3202", "jp_3453", "qw_11112", "qf_345", "dpql_3635", "jp_2435", "odql_13482", "qw_1298", "sfq_15917", "odql_8493", "qg_357", "sfq_5739", "qg_1580", "odql_5501", "dpql_151", "qg_1800", "bb_7364", "qw_16265", "sfq_6175", "jp_1572", "qw_15027", "qz_6596", "odql_7507", "sfq_17997", "qb_3940", "qw_14565", "odql_7762", "qw_3207", "sfq_2355", "sfq_14806", "sfq_10687", "odql_8306", "sfq_16554", "odql_15094", "bb_6092", "qg_2424", "tb_299", "qb_9584", "sfq_7904", "sfq_10502", "tb_1911", "qg_3056", "bt_1811", "bt_371", "qz_5459", "qb_4163", "qb_8707", "odql_14527", "qg_2316", "dpql_1746", "qb_6635", "odql_2749", "odql_4794", "tc_719", "bt_1604", "qf_2264", "sfq_17840", "qb_2148", "dpql_5194", "tc_1944", "qw_15528", "odql_2965", "sfq_14668", "qw_10673", "dpql_4666", "jp_1502", "sfq_6752"]}, {"condition": "dp_sigma_0p5", "score": "discrete_semantic_entropy", "threshold": 0.6108643020548935, "abstain_all": false, "risk_target": 0.1, "selection_scope": "same development data; exploratory", "eligible_ids": ["odql_7980", "qg_749", "odql_11176", "sfq_20838", "sfq_2187", "bb_8725", "qw_10107", "odql_11899", "sfq_8726", "odql_10108", "odql_3325", "tb_1852", "qg_152", "sfq_4423", "sfq_10592", "sfq_18424", "odql_2963", "sfq_12277", "sfq_17015", "bb_9117", "qz_1730", "sfq_23986", "qf_2590", "jp_2935", "sfq_7702", "sfq_23359", "odql_13834", "qg_774", "odql_7981", "wh_2615", "qw_10089", "qw_4795", "dpql_4823", "qb_1953", "qb_292", "odql_9495", "odql_11469", "bb_3733", "odql_14999", "sfq_1917", "qg_527", "qf_3609", "sfq_22986", "sfq_21768", "qb_2880", "bb_2139", "qw_45", "qw_3576", "sfq_6988", "qb_2169", "qf_387", "sfq_3731", "tb_83", "qg_633", "wh_267", "odql_14829", "sfq_15164", "qw_4450", "qb_4066", "qb_3535", "bt_1378", "qg_504", "bt_61", "bb_7375", "bb_1588", "qb_3775", "sfq_15066", "dpql_782", "odql_1772", "qz_6456", "odql_8729", "qf_2569", "odql_8774", "bt_4062", "wh_4082", "qw_2489", "sfq_24884", "qb_8477", "dpql_1994", "odql_13686", "bb_5892", "qg_2735", "odql_2476", "bt_3047", "odql_8663", "qb_4792", "dpql_3600", "sfq_23513", "bt_4028", "sfq_4588", "dpql_2210", "bb_6465", "qb_38", "qz_4435", "odql_4457", "odql_9352", "sfq_19745", "bb_2691", "odql_6624", "qb_8289", "qw_12650", "bt_1819", "sfq_11945", "qz_2822", "qb_7094", "jp_4067", "sfq_6665", "bt_434", "odql_136", "odql_6494", "qz_1047", "qz_4231", "qw_6139", "bt_3299", "qf_277", "odql_249", "dpql_1447", "sfq_13640", "dpql_580", "qf_2048", "dpql_3467", "qb_3556", "qw_2843", "sfq_19473", "jp_4032", "qw_7594", "sfq_3202", "jp_3453", "qw_11112", "qf_345", "dpql_3635", "jp_2435", "odql_13482", "qw_1298", "sfq_15917", "odql_8493", "qg_357", "sfq_5739", "qg_1580", "odql_5501", "dpql_151", "qg_1800", "bb_7364", "qw_16265", "sfq_6175", "jp_1572", "qw_15027", "qz_6596", "odql_7507", "sfq_17997", "qb_3940", "qw_14565", "odql_7762", "qw_3207", "sfq_2355", "sfq_14806", "sfq_10687", "odql_8306", "sfq_16554", "odql_15094", "bb_6092", "qg_2424", "tb_299", "qb_9584", "sfq_7904", "sfq_10502", "tb_1911", "qg_3056", "bt_1811", "bt_371", "qz_5459", "qb_4163", "qb_8707", "odql_14527", "qg_2316", "dpql_1746", "qb_6635", "odql_2749", "odql_4794", "tc_719", "bt_1604", "qf_2264", "sfq_17840", "qb_2148", "dpql_5194", "tc_1944", "qw_15528", "odql_2965", "sfq_14668", "qw_10673", "dpql_4666", "jp_1502", "sfq_6752"]}], "adapter_file_hashes": {"standard_poisson": "b8eb28f3bd48c1cd108404987289cead6c3346559ac8f52ac89b067afd76916c", "clipping_poisson": "44d6650efb5c38f8f8ab6ac803ea4301cb1e345f8998d1882c4904d7e2ef6933", "dp_sigma_0p1": "8df8f6ad2091d346b7fc13f9733a67a197176ea5495f74a1bb6452ca9cda8e1d", "dp_sigma_0p25": "954793528c402b95aa4f87cc2c01e9c2470a584c206c87a376f1d542316e248e", "dp_sigma_0p5": "fc73ce1a94d3de4ef1455683af4773f8d0a192576d51943a8e659d37ae623463"}, "development_report": "eb31652c708e", "threshold_file_sha256": "67b0d6b0490b3dac5fc8659fc2f235cd9bdda82034c3e41c59dc2cb6509f629c"}}')
config=read(TRAIN/'suite_config.json');common=config['common']
assert fp(config)==FROZEN[str(SEED)]['suite_config_hash'],'Wrong training run or settings.'
assert common['initialization_seed']==SEED
summaries={r['condition']:r for r in read(TRAIN/'suite_summary.json')}
assert set(summaries)==set(CONDITIONS)
assert all(r['optimizer_steps']==63 and r['adapter_reload_verified'] and r['all_finite_checks_passed'] for r in summaries.values())
assert len({r['initial_adapter_hash'] for r in summaries.values()})==1
MODEL_ID=common['model_id'];REVISION=common['model_revision'];shots=common['public_demonstrations']
splits=read(PROJECT/'data/qa_eval_splits_v1.json');assert fp(splits)==common['evaluation_split_hash']
dev=splits['test']  # Legacy variable name; ALL downstream rows are TEST questions.
assert len(dev)==1000 and len({r['question_id'] for r in dev})==1000
assert not ({r['question_id'] for r in dev}&{r['question_id'] for r in splits['dev']})
adapters={c:TRAIN/c/'adapter' for c in CONDITIONS}
adapter_hashes={c:fh(adapters[c]/'adapter_model.safetensors') for c in CONDITIONS}
adapter_configs={c:fh(adapters[c]/'adapter_config.json') for c in CONDITIONS}
assert adapter_hashes==FROZEN[str(SEED)]['adapter_file_hashes'], 'Adapters differ from completed development runs.'
NLI_ID='MoritzLaurer/DeBERTa-v3-base-mnli-fever-anli'
nli_pin=read(PROJECT/'results/llama31_8b_low_noise_uq_seed123_v1/nli_revision.json')
assert nli_pin['model_id']==NLI_ID;NLI_REV=nli_pin['revision']
versions={p:metadata.version(p) for p in ['torch','transformers','peft','accelerate','tokenizers','sentencepiece','scikit-learn','numpy','pandas']}
protocol={'version':'low_noise_heldout_v1','conditions':CONDITIONS,'scores':SCORES,
 'test_hash':fp(dev),'threshold_sources':FROZEN,'nli':nli_pin,
 'clustering':'all_members_bidirectional_entailment_v1','empty_samples':'separate_no_answer_cluster_v1',
 'samples':10,'temperature':.7,'top_p':.9,'top_k':0,'max_new_tokens':32,
 'sample_seed':'sha256(123:question_id) first8 hex; reset per question',
 'risk_target':.1,'threshold_rule':'frozen development threshold; accept score <= threshold; whole ties',
 'prior_exposure':'same test split previously evaluated with Qwen; not untouched independent confirmation',
 'label_policy':'human overrides AI; flags require human adjudication; shared validity across seeds',
 'primary_comparisons':'nine noisy-minus-clipping AUROC contrasts; paired question bootstrap, seed-mean estimand, Bonferroni family of nine; conditional on fitted adapters',
 'secondary':'semantic-minus-token scores; descriptive unless separately specified; coverage/risk at frozen thresholds'}
immutable(protocol,SHARED/'frozen_protocol.json')
identity={'protocol_hash':fp(protocol),'suite_config_hash':fp(config),'seed':SEED,
 'adapter_hashes':adapter_hashes,'adapter_config_hashes':adapter_configs,'versions':versions}
# ANALYSIS ONLY: use existing generation and scoring caches.
saved_identity = read(OUT / 'evaluation_identity.json')

# All experiment settings and adapter hashes must still match.
assert {
    k: v for k, v in saved_identity.items() if k != 'versions'
} == {
    k: v for k, v in identity.items() if k != 'versions'
}, 'Experiment settings changed. Stop and investigate.'

# Keep the numerical analysis environment consistent.
for package in ['numpy', 'pandas', 'scikit-learn']:
    assert versions[package] == saved_identity['versions'][package], (
        f'{package} changed: saved={saved_identity["versions"][package]}, '
        f'current={versions[package]}'
    )

BINDING = fp(saved_identity)
expected_ids = {r['question_id'] for r in dev}

for condition in CONDITIONS:
    records_file = records_path(condition)
    semantic_file = semantic_path(condition)
    assert records_file.exists() and semantic_file.exists(), (
        f'Missing saved scores for {condition}'
    )

    cache = read(records_file)
    assert cache['binding'] == BINDING
    records = cache['records']
    assert len(records) == len(expected_ids)
    assert {r['question_id'] for r in records} == expected_ids
    assert all(
        'greedy' in r
        and 'mean_token_entropy' in r
        and 'mean_answer_nll' in r
        and len(r.get('samples', [])) == 10
        for r in records
    ), f'Incomplete saved scores: {condition}'

    semantic = read(semantic_file)
    assert semantic['binding'] == {
        'evaluation': BINDING,
        'records_hash': fh(records_file),
    }
    assert len(semantic['records']) == len(expected_ids)
    assert {
        r['question_id'] for r in semantic['records']
    } == expected_ids
    assert all(
        'discrete_semantic_entropy' in r
        for r in semantic['records']
    )

# Record this analysis environment separately.
save(
    {
        'mode': 'analysis_only',
        'generation_binding': BINDING,
        'generation_versions': saved_identity['versions'],
        'analysis_versions': versions,
    },
    OUT / f'analysis_environment_{fp(versions)[:12]}.json',
)

print('Saved caches verified. Continue directly to Section 6.')
THRESHOLDS={(r['condition'],r['score']):r for r in FROZEN[str(SEED)]['thresholds']}
assert len(THRESHOLDS)==15
greedy={}
for c in CONDITIONS:
    path=records_path(c)
    cache=read(path) if path.exists() else {'binding':BINDING,'records':[]}
    assert cache['binding']==BINDING
    greedy[c]={r['question_id']:r['greedy'] for r in cache['records'] if 'greedy' in r}
print('TEST seed',SEED,'| questions',len(dev),'| output',OUT)
print('Thresholds frozen from development report:',FROZEN[str(SEED)]['development_report'])


## 2 · Generate test answers and uncertainty samples
This is the long GPU stage. It saves greedy answers and ten samples per question. Resume with Section 1 then this section after disconnecting.


In [ ]:
from google.colab import userdata
from getpass import getpass
from transformers import AutoTokenizer,AutoModelForCausalLM,GenerationConfig,StoppingCriteria,StoppingCriteriaList,set_seed
from peft import PeftModel,get_peft_model_state_dict
assert torch.cuda.is_available() and 'L4' in torch.cuda.get_device_name(0) and torch.cuda.is_bf16_supported()
try:HF_TOKEN=userdata.get('HF_TOKEN')
except Exception:HF_TOKEN=getpass('Hugging Face token (hidden): ')
assert HF_TOKEN

def prompt_ids(question):
    text='\n\n'.join('Question: '+r['question']+'\nAnswer: '+r['answer'].strip() for r in shots)
    text+='\n\nQuestion: '+question+'\nAnswer:'
    return [tokenizer.bos_token_id]+tokenizer.encode(text,add_special_tokens=False)
class StopAtNewline(StoppingCriteria):
    def __init__(self,start):self.start=start
    def __call__(self,input_ids,scores,**kwargs):
        return torch.tensor(['\n' in tokenizer.decode(input_ids[0,self.start:],skip_special_tokens=True)],device=input_ids.device)

def answer_prefix(generated,eos_ids):
    kept=[]; mixed=False
    for t in generated:
        if t in eos_ids:break
        previous=tokenizer.decode(kept,skip_special_tokens=True)
        candidate=tokenizer.decode(kept+[t],skip_special_tokens=True)
        if '\n' in candidate:
            before=candidate.split('\n',1)[0]
            # Keep an indivisible boundary token only when it contributes answer text.
            contribution=before[len(previous):] if before.startswith(previous) else before
            if contribution.strip():kept.append(t);mixed=True
            break
        kept.append(t)
    while kept:
        full=tokenizer.decode(kept,skip_special_tokens=True)
        shorter=tokenizer.decode(kept[:-1],skip_special_tokens=True)
        if full.strip()==shorter.strip():kept.pop()
        else:break
    return kept,mixed

@torch.inference_mode()
def score_greedy(model,item,saved,eos_ids):
    ids,mixed=answer_prefix(saved['generated_token_ids'],eos_ids)
    assert ids and saved['prediction'].strip(), 'Empty greedy answer needs investigation.'
    assert tokenizer.decode(saved['generated_token_ids'],skip_special_tokens=True).split('\n',1)[0].strip()==saved['prediction']
    prefix=prompt_ids(item['question'])
    x=torch.tensor([prefix+ids],device='cuda')
    logits=model(input_ids=x,attention_mask=torch.ones_like(x),use_cache=False).logits[0,len(prefix)-1:len(prefix)-1+len(ids)].float()
    assert torch.isfinite(logits).all()
    lp=torch.log_softmax(logits,dim=-1);prob=lp.exp()
    target=torch.tensor(ids,device='cuda')
    entropy=-(prob*lp).sum(-1).mean().item()
    nll=-lp.gather(1,target[:,None]).mean().item()
    assert math.isfinite(entropy) and math.isfinite(nll)
    return {'mean_token_entropy':entropy,'mean_answer_nll':nll,'scored_token_ids':ids,
            'mixed_answer_newline_token':mixed}

@torch.inference_mode()
def sample_answer(model,item,gen_config,eos_ids):
    ids=prompt_ids(item['question']);x=torch.tensor([ids],device='cuda')
    out=model.generate(input_ids=x,attention_mask=torch.ones_like(x),generation_config=gen_config,
        stopping_criteria=StoppingCriteriaList([StopAtNewline(len(ids))]))[0,len(ids):].tolist()
    raw=tokenizer.decode(out,skip_special_tokens=True);answer=raw.split('\n',1)[0].strip()
    ended=bool(out) and out[-1] in eos_ids
    return {'text':answer,'raw_continuation':raw,'generated_token_ids':out,
            'empty':not bool(answer),'hit_length_limit':len(out)>=32 and not ended and '\n' not in raw}

def compute_condition(c):
    global tokenizer
    path=records_path(c)
    cache=read(path) if path.exists() else {'binding':BINDING,'condition':c,'records':[]}
    assert cache['binding']==BINDING and cache['condition']==c
    by_id={r['question_id']:r for r in cache['records']}
    assert len(by_id)==len(cache['records']) and set(by_id)<={r['question_id'] for r in dev}
    for r in cache['records']:
        assert r['source_hash']==fp(next(x for x in dev if x['question_id']==r['question_id']))
        assert 'samples' not in r or len(r['samples'])==10
    if len(by_id)==len(dev) and all('mean_answer_nll' in r and len(r.get('samples',[]))==10 for r in cache['records']):return
    tokenizer=AutoTokenizer.from_pretrained(MODEL_ID,revision=REVISION,token=HF_TOKEN)
    tokenizer.pad_token=tokenizer.eos_token
    base=AutoModelForCausalLM.from_pretrained(MODEL_ID,revision=REVISION,token=HF_TOKEN,
        torch_dtype=torch.bfloat16,device_map={'':'cuda:0'},attn_implementation='sdpa',low_cpu_mem_usage=True)
    model=PeftModel.from_pretrained(base,adapters[c],is_trainable=False).eval()
    model.config.use_cache=True
    eos_ids=model.generation_config.eos_token_id or tokenizer.eos_token_id
    if isinstance(eos_ids,int):eos_ids=[eos_ids]
    assert eos_ids==read(PRE/'tokenizer_identity.json')['eos_ids']
    gen_config=GenerationConfig(do_sample=True,temperature=.7,top_p=.9,top_k=0,num_beams=1,
        max_new_tokens=32,eos_token_id=eos_ids,pad_token_id=tokenizer.pad_token_id,
        bos_token_id=tokenizer.bos_token_id,repetition_penalty=1.0,use_cache=True)
    for item in tqdm(dev,desc=c):
        q=item['question_id']
        if q not in by_id:
            by_id[q]={'question_id':q,'source_hash':fp(item)};cache['records'].append(by_id[q])
        row=by_id[q]
        if 'greedy' not in row:
            gconfig=GenerationConfig(do_sample=False,num_beams=1,max_new_tokens=32,
                eos_token_id=eos_ids,pad_token_id=tokenizer.pad_token_id,
                bos_token_id=tokenizer.bos_token_id,repetition_penalty=1.0,use_cache=True)
            g=sample_answer(model,item,gconfig,eos_ids)
            row['greedy']={**g,'prediction':g['text']};save(cache,path)
        greedy[c][q]=row['greedy']
        if 'samples' not in row:
            seed=int(hashlib.sha256(f'123:{q}'.encode()).hexdigest()[:8],16)
            set_seed(seed)
            row['samples']=[sample_answer(model,item,gen_config,eos_ids) for _ in range(10)]
            row['sampling_seed']=seed;save(cache,path)
        assert len(row['samples'])==10
        if 'mean_answer_nll' not in row:
            row.update(score_greedy(model,item,greedy[c][q],eos_ids));save(cache,path)
    del model,base;gc.collect();torch.cuda.empty_cache()
for c in CONDITIONS:compute_condition(c)
del HF_TOKEN
print('Sampling and token scoring complete. No comparative correctness metrics displayed.')


## 3 · Export blinded review and shared question validity
Finish Section 2 first. Prior question-validity decisions may be reused only when their original question and reference text match. No condition, uncertainty or automatic correctness labels enter the review ZIP.


In [ ]:
assert all(len(greedy[c])==len(dev) for c in CONDITIONS), 'Finish Section 2.'
salt_path=OUT/'review_salt_private.json'
if not salt_path.exists():save({'salt':secrets.token_hex(32)},salt_path)
salt=read(salt_path)['salt']
expected={};mapping=[]
for c in CONDITIONS:
    for item in dev:
        row=greedy[c][item['question_id']]
        aliases=list(dict.fromkeys([item['reference_answer'],*item['accepted_answers']]))
        key=[item['question_id'],item['question'],row['prediction'],item['reference_answer'],aliases]
        rid=fp([salt,key])
        expected[rid]={'review_id':rid,'question_id':item['question_id'],'question':item['question'],
            'prediction':row['prediction'],'reference_answer':item['reference_answer'],
            'accepted_answers':json.dumps(aliases,ensure_ascii=False)}
        mapping.append({'condition':c,'question_id':item['question_id'],'review_id':rid})
expected_review=pd.DataFrame([expected[k] for k in sorted(expected)])
review_path=OUT/'blind_answer_review.csv'
editable=['reviewed_correct','reviewer_type','reviewer','review_note','needs_adjudication','human_correct','human_reviewer','human_note']
if review_path.exists():
    current=pd.read_csv(review_path,keep_default_na=False,dtype=str)
    assert current.review_id.is_unique and set(current.review_id)==set(expected_review.review_id)
    current=current.set_index('review_id').loc[expected_review.review_id].reset_index()
    pd.testing.assert_frame_equal(current[list(expected_review.columns)],expected_review,check_dtype=False)
else:
    current=expected_review.copy()
    for col in editable:current[col]=''
    current.to_csv(review_path,index=False)
immutable(mapping,OUT/'review_mapping_private.json')
validity_path=SHARED/'test_question_validity.csv'
if not validity_path.exists():
    validity=pd.DataFrame(dev)[['question_id','question','reference_answer']].copy()
    for col in ['decision','reason','reviewer']:validity[col]=''
    prior_path=PROJECT/'results/final_test1000_v1/test_question_validity.csv'
    if prior_path.exists():
        prior=pd.read_csv(prior_path,keep_default_na=False,dtype=str)
        assert prior.question_id.is_unique and set(prior.question_id)==set(validity.question_id)
        prior=prior.set_index('question_id').loc[validity.question_id].reset_index()
        pd.testing.assert_frame_equal(prior[['question_id','question','reference_answer']],validity[['question_id','question','reference_answer']],check_dtype=False)
        if prior.decision.isin(['include','hold']).all() and prior.reviewer.str.strip().ne('').all():
            for col in ['decision','reason','reviewer']:validity[col]=prior[col].tolist()
            print('Reused prior test-question validity; original text verified.')
    validity.to_csv(validity_path,index=False)
# Validate question identities without requiring completed decisions yet.
v=pd.read_csv(validity_path,keep_default_na=False,dtype=str)
assert v.question_id.is_unique and set(v.question_id)=={x['question_id'] for x in dev}
v=v.set_index('question_id').loc[[x['question_id'] for x in dev]].reset_index()
pd.testing.assert_frame_equal(v[['question_id','question','reference_answer']],pd.DataFrame(dev)[['question_id','question','reference_answer']],check_dtype=False)
review_zip=OUT/f'llama_low_noise_test_review_seed{SEED}.zip'
with zipfile.ZipFile(review_zip,'w',zipfile.ZIP_DEFLATED) as z:
    for p in [review_path,validity_path]:z.write(p,p.name)
    z.writestr('README.txt','Review only the greedy predictions. Accept unambiguous aliases and harmless spelling variations; reject contradictions, refusals, insufficient specificity, and repetitions. Judge truncation by visible content. Hold invalid questions consistently across all conditions. Identify AI-assisted labels; do not claim a human audit without one. Preserve all IDs and original text. Fill review_note with rationale. Identical answers across conditions share this review. Return the two CSVs unchanged in structure. No condition mapping or uncertainty scores belong in this reviewer bundle.')
print('Distinct review answers:',len(current))
print('Review bundle:',review_zip)


## 4 · Calculate semantic entropy
Original NLI method; no development-specific pair overrides. Empty sampled answers remain their own group.


In [ ]:
from transformers import AutoTokenizer,AutoModelForSequenceClassification
NLI_DEVICE='cuda' if torch.cuda.is_available() else 'cpu'
nli_tokenizer=AutoTokenizer.from_pretrained(NLI_ID,revision=NLI_REV)
nli_model=AutoModelForSequenceClassification.from_pretrained(NLI_ID,revision=NLI_REV,
    torch_dtype=torch.float32).to(NLI_DEVICE).eval()
nli_labels={int(i):label.lower() for i,label in nli_model.config.id2label.items()}
assert set(nli_labels.values())=={'entailment','neutral','contradiction'}


In [ ]:
@torch.inference_mode()
def compare_meanings(question, answer_a, answer_b):
    text_a = f"Question: {question}\nAnswer: {answer_a}"
    text_b = f"Question: {question}\nAnswer: {answer_b}"

    inputs = nli_tokenizer(
        [text_a, text_b],
        [text_b, text_a],
        padding=True,
        truncation=False,
        return_tensors="pt",
    ).to(NLI_DEVICE)

    if inputs["input_ids"].shape[1] > 512:
        raise ValueError("NLI pair too long; inspect before truncating.")

    logits = nli_model(**inputs).logits.float()
    probabilities = logits.softmax(dim=-1)
    predicted_ids = probabilities.argmax(dim=-1).tolist()

    forward_label = nli_labels[predicted_ids[0]]
    backward_label = nli_labels[predicted_ids[1]]

    return {
        "a_to_b": forward_label,
        "b_to_a": backward_label,
        "equivalent": (
            forward_label == "entailment"
            and backward_label == "entailment"
        ),
    }

import math

def calculate_semantic_entropy(question, answers):
    if not answers:
        raise ValueError("Missing sampled answers.")

    clusters = []
    cluster_ids = []
    pair_cache = {}

    def equivalent(a, b):
        if not a.strip() or not b.strip():
            return a.strip()==b.strip()
        # Identical text always belongs to the same meaning group.
        if a == b:
            return True

        key = tuple(sorted((a, b)))
        if key not in pair_cache:
            pair_cache[key] = compare_meanings(
                question, a, b
            )["equivalent"]

        return pair_cache[key]

    for answer in answers:
        answer = answer.strip()
        assigned_cluster = None

        for cluster_id, cluster in enumerate(clusters):
            distinct_members = list(dict.fromkeys(cluster))

            if all(equivalent(answer, member)
                   for member in distinct_members):
                assigned_cluster = cluster_id
                break

        if assigned_cluster is None:
            assigned_cluster = len(clusters)
            clusters.append([])

        clusters[assigned_cluster].append(answer)
        cluster_ids.append(assigned_cluster)

    probabilities = [
        len(cluster) / len(answers)
        for cluster in clusters
    ]

    entropy = -sum(p * math.log(p) for p in probabilities)

    return {
        "semantic_cluster_ids": cluster_ids,
        "semantic_clusters": clusters,
        "num_semantic_clusters": len(clusters),
        "discrete_semantic_entropy": entropy,
    }

In [ ]:
for c in CONDITIONS:
    source=read(records_path(c));assert source['binding']==BINDING and len(source['records'])==len(dev)
    binding={'evaluation':BINDING,'records_hash':fh(records_path(c))}
    path=semantic_path(c)
    cache=read(path) if path.exists() else {'binding':binding,'records':[]}
    assert cache['binding']==binding
    by_id={r['question_id']:r for r in cache['records']}
    assert len(by_id)==len(cache['records']) and set(by_id)<={r['question_id'] for r in dev}
    questions={r['question_id']:r['question'] for r in dev}
    for r in tqdm(source['records'],desc='NLI '+c):
        q=r['question_id']
        if q not in by_id:
            result=calculate_semantic_entropy(questions[q],[s['text'] for s in r['samples']])
            row={'question_id':q,**result};cache['records'].append(row);by_id[q]=row;save(cache,path)
    assert len(cache['records'])==len(dev)
    inspection=[]
    for row in cache['records']:
        for i,cluster in enumerate(row['semantic_clusters']):
            inspection.append({'question_id':row['question_id'],'question':questions[row['question_id']],
                'cluster_id':i,'samples':json.dumps(cluster,ensure_ascii=False)})
    pd.DataFrame(inspection).to_csv(OUT/(c+'_cluster_inspection.csv'),index=False)
del nli_model;gc.collect()
if torch.cuda.is_available():torch.cuda.empty_cache()
print('Semantic scores complete; cluster inspection files stay separate from reviewer bundle.')


## 5 · Download review bundle and stop
Send the review ZIP. Restore reviewed blind_answer_review.csv in the seed output folder and test_question_validity.csv in the shared test folder before Section 6.


In [ ]:
from google.colab import files
review_zip=OUT/f'llama_low_noise_test_review_seed{SEED}.zip'
assert review_zip.exists()
print('Answer review folder:',OUT)
print('Shared validity folder:',SHARED)
files.download(str(review_zip))


## 6 · After review: frozen-threshold test metrics
Run Section 1 then this section. No model loading. Risk is undefined when no answers are accepted. Test scores never select a new threshold. Per-question results enable a later paired bootstrap across all three seeds.


In [ ]:
from sklearn.metrics import roc_auc_score,average_precision_score
import matplotlib.pyplot as plt
v=pd.read_csv(SHARED/'test_question_validity.csv',keep_default_na=False,dtype=str)
assert v.question_id.is_unique and set(v.question_id)=={x['question_id'] for x in dev}
v=v.set_index('question_id').loc[[x['question_id'] for x in dev]].reset_index()
pd.testing.assert_frame_equal(v[['question_id','question','reference_answer']],pd.DataFrame(dev)[['question_id','question','reference_answer']],check_dtype=False)
assert v.decision.isin(['include','hold']).all(), 'Complete include/hold decisions for every question.'
assert v.reviewer.str.strip().ne('').all()
assert v.loc[v.decision=='hold','reason'].str.strip().ne('').all()
eligible=v.loc[v.decision=='include','question_id'].tolist();assert eligible
review=pd.read_csv(OUT/'blind_answer_review.csv',keep_default_na=False,dtype=str)
assert review.review_id.is_unique
salt=read(OUT/'review_salt_private.json')['salt']
expected={};mapping=[]
for c in CONDITIONS:
    for item in dev:
        pred=greedy[c][item['question_id']]['prediction'];aliases=list(dict.fromkeys([item['reference_answer'],*item['accepted_answers']]))
        rid=fp([salt,[item['question_id'],item['question'],pred,item['reference_answer'],aliases]])
        expected[rid]={'question_id':item['question_id'],'question':item['question'],'prediction':pred,
            'reference_answer':item['reference_answer'],'accepted_answers':json.dumps(aliases,ensure_ascii=False)}
        mapping.append({'condition':c,'question_id':item['question_id'],'review_id':rid})
assert mapping==read(OUT/'review_mapping_private.json')
assert set(review.review_id)==set(expected)
labels={};audit=[]
for r in review.to_dict('records'):
    assert all(r[k]==value for k,value in expected[r['review_id']].items()), 'Original review content changed.'
    if r['question_id'] not in eligible:continue
    assert r['reviewed_correct'] in ['0','1'] and r['reviewer_type'] in ['human','AI-assisted']
    assert r['reviewer'].strip() and r['review_note'].strip()
    assert r['needs_adjudication'] in ['0','1']
    human=r['human_correct'].strip()
    if human:
        assert human in ['0','1'] and r['human_reviewer'].strip() and r['human_note'].strip()
    assert r['needs_adjudication']=='0' or human, 'Resolve flagged answers before reporting.'
    label=int(human or r['reviewed_correct']);labels[r['review_id']]=label
    audit.append({'review_id':r['review_id'],'initial_label':int(r['reviewed_correct']),'final_label':label,
        'human_checked':bool(human) or r['reviewer_type']=='human','reviewer_type':r['reviewer_type']})
status='human-reviewed test labels' if all(r['human_checked'] for r in audit) else 'provisional AI-assisted test labels; human audit incomplete'
inputs={'evaluation':BINDING,'review':fh(OUT/'blind_answer_review.csv'),'validity':fh(SHARED/'test_question_validity.csv'),
        'score_files':{c:[fh(records_path(c)),fh(semantic_path(c))] for c in CONDITIONS}}
REPORT=OUT/('reports_'+fp(inputs)[:12]);REPORT.mkdir(exist_ok=True)
immutable(inputs,REPORT/'analysis_inputs.json')

def whole_tie_curve(scores,errors):
    order=np.argsort(scores,kind='stable');s=np.asarray(scores)[order];e=np.asarray(errors)[order]
    end=np.r_[np.flatnonzero(s[:-1]!=s[1:]),len(s)-1]
    counts=end+1;coverage=counts/len(s);risk=np.cumsum(e)[end]/counts
    aurc=float(np.sum(np.diff(np.r_[0.,coverage])*risk))
    return s[end],coverage,risk,aurc

outcomes=[];curves=[];thresholds=[];zero=[];per_question=[]
for c in CONDITIONS:
    record_cache=read(records_path(c));assert record_cache['binding']==BINDING
    semantic_cache=read(semantic_path(c));assert semantic_cache['binding']=={'evaluation':BINDING,'records_hash':fh(records_path(c))}
    records={r['question_id']:r for r in record_cache['records']};sem={r['question_id']:r for r in semantic_cache['records']}
    assert len(records)==len(record_cache['records'])==len(sem)==len(semantic_cache['records'])==len(dev)
    link={r['question_id']:r['review_id'] for r in mapping if r['condition']==c}
    y=np.array([1-labels[link[q]] for q in eligible]);acc=float(1-y.mean())
    for score in SCORES:
        x=np.array([sem[q][score] if score=='discrete_semantic_entropy' else records[q][score] for q in eligible])
        assert np.isfinite(x).all()
        auc=float(roc_auc_score(y,x)) if len(set(y))==2 else None
        ap=float(average_precision_score(y,x)) if len(set(y))==2 else None
        ts,cov,risk,aurc=whole_tie_curve(x,y)
        frozen_threshold=THRESHOLDS[c,score]
        chosen=frozen_threshold['threshold']
        accepted=np.zeros(len(y),dtype=bool) if frozen_threshold['abstain_all'] else x<=float(chosen)
        coverage=float(accepted.mean())
        accepted_risk=float(y[accepted].mean()) if accepted.any() else None
        for q,score_value,error,take in zip(eligible,x,y,accepted):
            per_question.append({'seed':SEED,'condition':c,'question_id':q,'score':score,
              'value':float(score_value),'incorrect':int(error),'accepted':bool(take)})
        outcomes.append({'condition':c,'score':score,'n':len(y),'accuracy':acc,'error_prevalence':float(y.mean()),
            'AUROC':auc,'AP':ap,'AURC_tie_step':aurc,
            'test_coverage_at_frozen_threshold':coverage,
            'test_risk_at_frozen_threshold':accepted_risk,
            'accounted_epsilon':summaries[c]['accounted_epsilon'],'review_status':status})
        thresholds.append({'condition':c,'score':score,'threshold':chosen,'abstain_all':frozen_threshold['abstain_all'],
            'risk_target':.10,'selection_scope':'development only; frozen before test evaluation','eligible_ids':eligible})
        curves.extend({'condition':c,'score':score,'threshold':float(t),'coverage':float(a),'risk':float(b)} for t,a,b in zip(ts,cov,risk))
    mask=np.array([sem[q]['discrete_semantic_entropy']==0 for q in eligible])
    zero.append({'condition':c,'zero_entropy_n':int(mask.sum()),'coverage':float(mask.mean()),
                 'risk':float(y[mask].mean()) if mask.any() else None})
frame=pd.DataFrame(outcomes)
lookup={(r['condition'],r['score']):r['AUROC'] for r in outcomes}
contrasts=[]
for c in CONDITIONS:
    if not c.startswith('dp_'):continue
    for score in SCORES:
        a=lookup[c,score];b=lookup['clipping_poisson',score]
        contrasts.append({'comparison':c+' minus clipping_poisson','score':score,'delta_AUROC':None if a is None or b is None else a-b})
advantages=[]
for c in CONDITIONS:
    for comparator in SCORES[:2]:
        vals=[lookup[c,'discrete_semantic_entropy'],lookup[c,comparator],lookup['clipping_poisson','discrete_semantic_entropy'],lookup['clipping_poisson',comparator]]
        advantages.append({'condition':c,'comparator':comparator,
            'semantic_advantage':None if any(v is None for v in vals) else vals[0]-vals[1],
            'change_vs_clipping':None if any(v is None for v in vals) else (vals[0]-vals[1])-(vals[2]-vals[3])})
frame.insert(0,'seed',SEED)
frame.to_csv(REPORT/'outcomes.csv',index=False)
pd.DataFrame(per_question).to_csv(REPORT/'per_question_scores.csv',index=False)
save(protocol,REPORT/'frozen_protocol.json')
pd.DataFrame(curves).to_csv(REPORT/'risk_coverage_curves.csv',index=False)
pd.DataFrame(contrasts).to_csv(REPORT/'noise_vs_clipping_AUROC.csv',index=False)
pd.DataFrame(advantages).to_csv(REPORT/'semantic_advantage.csv',index=False)
pd.DataFrame(zero).to_csv(REPORT/'zero_entropy_diagnostics.csv',index=False)
pd.DataFrame(audit).to_csv(REPORT/'review_provenance.csv',index=False)
v.to_csv(REPORT/'question_validity.csv',index=False)
save(thresholds,REPORT/'applied_development_thresholds.json')
for score in SCORES:
    fig,ax=plt.subplots(figsize=(7,4))
    for c in CONDITIONS:
        a=[r for r in curves if r['condition']==c and r['score']==score]
        ax.plot([r['coverage'] for r in a],[r['risk'] for r in a],marker='.',markersize=2,label=c)
    ax.axhline(.10,color='gray',linestyle='--');ax.set(xlabel='Coverage',ylabel='Observed error',title=score+' (test; descriptive curve)',ylim=(0,1),xlim=(0,1))
    ax.legend(fontsize=7);fig.tight_layout();fig.savefig(REPORT/(score+'_risk_coverage.png'),dpi=150);plt.close(fig)
(REPORT/'README.txt').write_text(status+'\nSingle-seed Llama test results on a split previously used for Qwen. Thresholds frozen from development; no guarantee of test risk <=10%. No significance inference. AURC uses whole ties and right-endpoint step integration. Plots connect attainable points only for readability. Epsilon is standalone research mechanism accounting, not a joint private release. Human review changes are recorded through versioned input hashes.')
report_zip=OUT/(f'llama_low_noise_test_seed{SEED}_reports_'+fp(inputs)[:12]+'.zip')
with zipfile.ZipFile(report_zip,'w',zipfile.ZIP_DEFLATED) as z:
    for p in REPORT.iterdir():
        if p.is_file():z.write(p,p.name)
display(frame)
print('Send:',report_zip)
from google.colab import files
files.download(str(report_zip))
